In [1]:


%pip -q install "datasets>=3.0,<5.0" "huggingface_hub>=0.25" numpy pandas scikit-learn tqdm z3-solver

import os
import sys
import re
import json
import time
import math
import random
import platform
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict, Counter, deque
from dataclasses import dataclass, field
from itertools import product
from typing import Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import hf_hub_download, list_repo_files

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("/kaggle/working/AI_Assignment_2026")
DATA_DIR = PROJECT_DIR / "data"
NORMALIZED_DIR = DATA_DIR / "normalized"
OUTPUT_DIR = PROJECT_DIR / "outputs"
REPORT_DIR = PROJECT_DIR / "report"
CACHE_DIR = PROJECT_DIR / "cache"

for folder in [PROJECT_DIR, DATA_DIR, NORMALIZED_DIR, OUTPUT_DIR, REPORT_DIR, CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_Token") or os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        try:
            HF_TOKEN = secrets.get_secret("HF_Token")
        except Exception:
            HF_TOKEN = secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "Hugging Face token not found. In Kaggle, open Add-ons → Secrets, "
        "add a secret named HF_Token containing your read token, and enable "
        "notebook access to that secret. Do not hard-code or publish the token."
    )

# Repositories specified/used by the assignment project.
RULETAKER_REPO = "jise/ruletaker"
FOLIO_REPO = "tasksource/folio"
COUNTERBENCH_REPO = "CounterBench/CounterBench"

print("=" * 72)
print("AI ASSIGNMENT 2026 M26CSA005")
print("=" * 72)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project:", PROJECT_DIR)
print("Hugging Face token available:", bool(HF_TOKEN))
print("RuleTaker:", RULETAKER_REPO)
print("FOLIO:", FOLIO_REPO)
print("CounterBench:", COUNTERBENCH_REPO)

Note: you may need to restart the kernel to use updated packages.
AI ASSIGNMENT 2026 M26CSA005
Python: 3.13.18
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Project: /kaggle/working/AI_Assignment_2026
Hugging Face token available: True
RuleTaker: jise/ruletaker
FOLIO: tasksource/folio
CounterBench: CounterBench/CounterBench


In [2]:
# ============================================================
# SECTION 2 — DATA DOWNLOADING FROM HUGGING FACE AND NORMALIZATION
# ============================================================

from datasets import load_dataset
from huggingface_hub import hf_hub_download, list_repo_files
import json
import pandas as pd

def _to_frame(dataset_split):
    """Convert a Hugging Face Dataset split to a pandas DataFrame."""
    return dataset_split.to_pandas()

def _read_json_records(path):
    """Read a JSON list/dict or JSONL file downloaded from the Hub."""
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    if not raw:
        raise ValueError(f"Downloaded dataset file is empty: {path}")
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        obj = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        # CounterBench source files are expected to be list-like; handle
        # common dictionary wrappers without silently flattening scalars.
        for key in ("data", "records", "questions", "models"):
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])
        return pd.DataFrame(list(obj.values()))
    raise TypeError(f"Unsupported JSON structure in {path}: {type(obj).__name__}")

print("=" * 72)
print("[1/3] Downloading RuleTaker from Hugging Face...")
ruletaker_data = load_dataset(RULETAKER_REPO, token=HF_TOKEN)
RULETAKER_DF = {}
for split_name, split in ruletaker_data.items():
    frame = _to_frame(split)
    for col in ["context", "statement", "reasoning", "flag"]:
        if col in frame.columns:
            frame[col] = frame[col].astype(str).str.strip()
    if "flag" in frame.columns:
        frame["flag"] = frame["flag"].astype(str).str.strip().str.capitalize()
    RULETAKER_DF[split_name] = frame
    frame.to_json(
        NORMALIZED_DIR / f"ruletaker_{split_name}.jsonl",
        orient="records", lines=True, force_ascii=False
    )
    print(f"RuleTaker/{split_name}: {len(frame):,} rows")

print("\n[2/3] Downloading FOLIO from Hugging Face...")
folio_data = load_dataset(FOLIO_REPO, token=HF_TOKEN)
FOLIO_DF = {}
for split_name, split in folio_data.items():
    frame = _to_frame(split)
    for col in ["premises", "premises-FOL", "conclusion", "conclusion-FOL", "label"]:
        if col in frame.columns:
            frame[col] = frame[col].astype(str).str.strip()
    FOLIO_DF[split_name] = frame
    frame.to_json(
        NORMALIZED_DIR / f"folio_{split_name}.jsonl",
        orient="records", lines=True, force_ascii=False
    )
    print(f"FOLIO/{split_name}: {len(frame):,} rows")

print("\n[3/3] Downloading CounterBench JSON files from Hugging Face...")
counterbench_files = [
    "data_balanced_alpha_V1.json",
    "data_balanced_backdoor_V2.json",
    "meta_model_alpha_V1.json",
    "meta_model_backdoor_V2.json",
]
repo_files = set(list_repo_files(
    COUNTERBENCH_REPO, repo_type="dataset", token=HF_TOKEN
))
missing_repo_files = [name for name in counterbench_files if name not in repo_files]
if missing_repo_files:
    raise FileNotFoundError(
        "CounterBench repository is missing expected files: "
        + ", ".join(missing_repo_files)
    )

counterbench_paths = {}
for filename in counterbench_files:
    counterbench_paths[filename] = hf_hub_download(
        repo_id=COUNTERBENCH_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
        cache_dir=str(CACHE_DIR),
    )
    print("Downloaded:", filename)

alpha_questions = _read_json_records(counterbench_paths["data_balanced_alpha_V1.json"])
backdoor_questions = _read_json_records(counterbench_paths["data_balanced_backdoor_V2.json"])
alpha_metadata = _read_json_records(counterbench_paths["meta_model_alpha_V1.json"])
backdoor_metadata = _read_json_records(counterbench_paths["meta_model_backdoor_V2.json"])

required_q = {"question", "given_info", "answer", "type", "question_id", "meta"}
required_m = {"model_id", "background", "variable_mapping", "nonsense"}
for name, frame in {
    "CounterBench Alpha questions": alpha_questions,
    "CounterBench Backdoor questions": backdoor_questions,
}.items():
    missing = required_q - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing required columns: {sorted(missing)}")
for name, frame in {
    "CounterBench Alpha metadata": alpha_metadata,
    "CounterBench Backdoor metadata": backdoor_metadata,
}.items():
    missing = required_m - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing required columns: {sorted(missing)}")

def _extract_cb_metadata(frame):
    frame = frame.copy()
    frame["graph_id"] = frame["meta"].apply(
        lambda x: x.get("graph_id") if isinstance(x, dict) else None
    )
    frame["model_id"] = frame["meta"].apply(
        lambda x: x.get("model_id") if isinstance(x, dict) else None
    )
    frame["model_id"] = pd.to_numeric(frame["model_id"], errors="coerce").astype("Int64")
    frame["question_id"] = frame["question_id"].astype(str)
    frame["answer"] = frame["answer"].astype(str).str.lower().str.strip()
    return frame

alpha_questions = _extract_cb_metadata(alpha_questions)
backdoor_questions = _extract_cb_metadata(backdoor_questions)
alpha_metadata["model_id"] = pd.to_numeric(alpha_metadata["model_id"], errors="raise").astype("Int64")
backdoor_metadata["model_id"] = pd.to_numeric(backdoor_metadata["model_id"], errors="raise").astype("Int64")

COUNTERBENCH_ALPHA = alpha_questions.merge(
    alpha_metadata, on="model_id", how="left",
    validate="many_to_one", suffixes=("", "_model"), indicator=True
)
COUNTERBENCH_BACKDOOR = backdoor_questions.merge(
    backdoor_metadata, on="model_id", how="left",
    validate="many_to_one", suffixes=("", "_model"), indicator=True
)
for dataset_name, frame in [
    ("Alpha", COUNTERBENCH_ALPHA), ("Backdoor", COUNTERBENCH_BACKDOOR)
]:
    unmatched = int((frame["_merge"] != "both").sum())
    if unmatched:
        raise ValueError(f"CounterBench {dataset_name}: {unmatched} rows lack model metadata.")
    frame.drop(columns=["_merge"], inplace=True)
    if not set(frame["answer"].unique()).issubset({"yes", "no"}):
        raise ValueError(f"Unexpected answer labels in CounterBench {dataset_name}.")

COUNTERBENCH_ALPHA.to_json(
    NORMALIZED_DIR / "counterbench_alpha.jsonl",
    orient="records", lines=True, force_ascii=False
)
COUNTERBENCH_BACKDOOR.to_json(
    NORMALIZED_DIR / "counterbench_backdoor.jsonl",
    orient="records", lines=True, force_ascii=False
)

# Preserve dataset provenance and exact Hub identifiers.
dataset_metadata = {
    "source": "Hugging Face Hub",
    "repositories": {
        "ruletaker": RULETAKER_REPO,
        "folio": FOLIO_REPO,
        "counterbench": COUNTERBENCH_REPO,
    },
    "ruleTaker_splits": {
        k: {"rows": len(v), "columns": list(v.columns)}
        for k, v in RULETAKER_DF.items()
    },
    "folio_splits": {
        k: {"rows": len(v), "columns": list(v.columns)}
        for k, v in FOLIO_DF.items()
    },
    "counterbench": {
        "alpha_rows": len(COUNTERBENCH_ALPHA),
        "backdoor_rows": len(COUNTERBENCH_BACKDOOR),
        "source_files": counterbench_files,
    },
}
with open(DATA_DIR / "dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2)

print("\n" + "=" * 72)
print("HUGGING FACE DOWNLOAD AND NORMALIZATION COMPLETED")
print("RuleTaker splits:", {k: len(v) for k, v in RULETAKER_DF.items()})
print("FOLIO splits:", {k: len(v) for k, v in FOLIO_DF.items()})
print("CounterBench Alpha:", len(COUNTERBENCH_ALPHA))
print("CounterBench Backdoor:", len(COUNTERBENCH_BACKDOOR))
print("Normalized data directory:", NORMALIZED_DIR)
print("Metadata:", DATA_DIR / "dataset_metadata.json")

[1/3] Downloading RuleTaker from Hugging Face...


README.md:   0%|          | 0.00/955 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  242kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.5kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (16 outputs left out)

meta_model_alpha_V1.json:   0%|          | 0.00/759k [00:00<?, ?B/s]

Downloaded: meta_model_alpha_V1.json


meta_model_backdoor_V2.json:   0%|          | 0.00/160k [00:00<?, ?B/s]

Downloaded: meta_model_backdoor_V2.json

HUGGING FACE DOWNLOAD AND NORMALIZATION COMPLETED
RuleTaker splits: {'train': 1000, 'test': 250, 'ood_test': 375}
FOLIO splits: {'train': 1001, 'validation': 203}
CounterBench Alpha: 1000
CounterBench Backdoor: 200
Normalized data directory: /kaggle/working/AI_Assignment_2026/data/normalized
Metadata: /kaggle/working/AI_Assignment_2026/data/dataset_metadata.json


In [3]:
# ============================================================
# SECTION 3 — LOAD AND VALIDATION OF NORMALIZED DATASETS
# ============================================================

def read_jsonl(filename):
    path = NORMALIZED_DIR / filename
    frame = pd.read_json(path, lines=True)
    if frame.empty:
        raise ValueError(f"{filename} is empty.")
    return frame

RULETAKER_DF = {
    "train": read_jsonl("ruletaker_train.jsonl"),
    "test": read_jsonl("ruletaker_test.jsonl"),
    "ood_test": read_jsonl("ruletaker_ood_test.jsonl"),
}
FOLIO_DF = {
    "train": read_jsonl("folio_train.jsonl"),
    "validation": read_jsonl("folio_validation.jsonl"),
}
COUNTERBENCH_ALPHA = read_jsonl("counterbench_alpha.jsonl")
COUNTERBENCH_BACKDOOR = read_jsonl("counterbench_backdoor.jsonl")

# Standardize only label whitespace/case; do not alter question semantics.
for split_name, frame in RULETAKER_DF.items():
    for column in ["context", "statement", "reasoning", "flag"]:
        if column in frame.columns:
            frame[column] = frame[column].astype(str).str.strip()
    if "flag" in frame.columns:
        frame["flag"] = frame["flag"].str.capitalize()

for split_name, frame in FOLIO_DF.items():
    for column in ["premises", "premises-FOL", "conclusion", "conclusion-FOL", "label"]:
        if column in frame.columns:
            frame[column] = frame[column].astype(str).str.strip()

for name, frame in [
    ("CounterBench Alpha", COUNTERBENCH_ALPHA),
    ("CounterBench Backdoor", COUNTERBENCH_BACKDOOR),
]:
    required = {"question", "given_info", "answer", "type", "question_id"}
    absent = required - set(frame.columns)
    if absent:
        raise ValueError(f"{name} is missing columns: {sorted(absent)}")
    frame["answer"] = frame["answer"].astype(str).str.strip().str.lower()
    frame["question_id"] = frame["question_id"].astype(str)

# Preserve the names expected by the original Alpha section.
alpha_questions = COUNTERBENCH_ALPHA.copy()
backdoor_questions = COUNTERBENCH_BACKDOOR.copy()
alpha_df = COUNTERBENCH_ALPHA.copy()

print("=" * 72)
print("DATASET SUMMARY")
print("=" * 72)
for split, frame in RULETAKER_DF.items():
    print(f"RuleTaker {split:10s}: {len(frame):5d}")
for split, frame in FOLIO_DF.items():
    print(f"FOLIO {split:14s}: {len(frame):5d}")
print(f"CounterBench Alpha     : {len(COUNTERBENCH_ALPHA):5d}")
print(f"CounterBench Backdoor  : {len(COUNTERBENCH_BACKDOOR):5d}")
print("\nCounterBench Backdoor labels:")
print(COUNTERBENCH_BACKDOOR["answer"].value_counts().to_dict())


DATASET SUMMARY
RuleTaker train     :  1000
RuleTaker test      :   250
RuleTaker ood_test  :   375
FOLIO train         :  1001
FOLIO validation    :   203
CounterBench Alpha     :  1000
CounterBench Backdoor  :   200

CounterBench Backdoor labels:
{'no': 105, 'yes': 95}


In [4]:

# ============================================================
# SECTION 4 — SHARED LOGICAL REPRESENTATION
# ============================================================

from dataclasses import dataclass, field
from typing import Optional, Iterable
from collections import defaultdict
import re

# ------------------------------------------------------------
# 1. Normalize logical text
# ------------------------------------------------------------

def normalize_text(text: str) -> str:
    """
    Normalize whitespace and casing for logical comparison.
    Preserve meaningful words and punctuation inside predicates.
    """
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[.!?]+$", "", text)
    return text.strip()


# ------------------------------------------------------------
# 2. Represent an atomic proposition
# ------------------------------------------------------------

@dataclass(frozen=True)
class Atom:
    """
    An atomic proposition represented by a predicate and arguments.

    Examples:
        Atom("rains", ())
        Atom("likes", ("john", "mary"))
    """
    predicate: str
    arguments: tuple = ()

    def __post_init__(self):
        object.__setattr__(
            self,
            "predicate",
            normalize_text(self.predicate)
        )
        object.__setattr__(
            self,
            "arguments",
            tuple(normalize_text(arg) for arg in self.arguments)
        )

    def __str__(self):
        if not self.arguments:
            return self.predicate

        return f"{self.predicate}({', '.join(self.arguments)})"


# ------------------------------------------------------------
# 3. Represent positive and negative literals
# ------------------------------------------------------------

@dataclass(frozen=True)
class Literal:
    atom: Atom
    negated: bool = False

    def __str__(self):
        return f"not {self.atom}" if self.negated else str(self.atom)


# ------------------------------------------------------------
# 4. Represent an implication rule
# ------------------------------------------------------------

@dataclass(frozen=True)
class LogicalRule:
    """
    A Horn-style implication:

        antecedents -> consequent

    Example:
        [rainy, cold] -> wears_coat
    """
    antecedents: tuple
    consequent: Literal
    rule_id: Optional[str] = None

    def __post_init__(self):
        object.__setattr__(
            self,
            "antecedents",
            tuple(self.antecedents)
        )

        if not all(
            isinstance(item, Literal)
            for item in self.antecedents
        ):
            raise TypeError("All antecedents must be Literal objects.")

        if not isinstance(self.consequent, Literal):
            raise TypeError("Consequent must be a Literal object.")

    def __str__(self):
        left = " AND ".join(str(x) for x in self.antecedents)
        return f"{left} -> {self.consequent}"


# ------------------------------------------------------------
# 5. Represent a knowledge base
# ------------------------------------------------------------

@dataclass
class KnowledgeBase:
    facts: set = field(default_factory=set)
    rules: list = field(default_factory=list)

    def add_fact(self, literal: Literal):
        if not isinstance(literal, Literal):
            raise TypeError("Knowledge-base facts must be Literal objects.")
        self.facts.add(literal)

    def add_rule(self, rule: LogicalRule):
        if not isinstance(rule, LogicalRule):
            raise TypeError("Expected a LogicalRule object.")
        self.rules.append(rule)

    def contains(self, literal: Literal) -> bool:
        return literal in self.facts

    def summary(self):
        return {
            "number_of_facts": len(self.facts),
            "number_of_rules": len(self.rules),
            "facts": sorted(str(fact) for fact in self.facts),
            "rules": [str(rule) for rule in self.rules]
        }


# ------------------------------------------------------------
# 6. Helper for creating simple propositional literals
# ------------------------------------------------------------

def make_literal(text: str) -> Literal:
    """
    Convert a simple text proposition into a Literal.

    This helper is intended for normalized propositional atoms.
    More complex first-order formulas will be handled by the
    dedicated FOLIO parser.
    """
    text = normalize_text(text)

    negated = False

    if text.startswith("not "):
        negated = True
        text = text[4:].strip()
    elif text.startswith("not(") and text.endswith(")"):
        negated = True
        text = text[4:-1].strip()

    if not text:
        raise ValueError("Cannot create a literal from empty text.")

    return Literal(
        atom=Atom(predicate=text),
        negated=negated
    )


# ------------------------------------------------------------
# 7. Test the representation
# ------------------------------------------------------------

test_kb = KnowledgeBase()

fact_1 = make_literal("The river is fast.")
fact_2 = make_literal("The wall dances gracefully.")

rule_1 = LogicalRule(
    antecedents=(fact_1,),
    consequent=make_literal("The river becomes famous."),
    rule_id="R1"
)

rule_2 = LogicalRule(
    antecedents=(
        make_literal("The river becomes famous"),
        fact_2
    ),
    consequent=make_literal("The river is interesting"),
    rule_id="R2"
)

test_kb.add_fact(fact_1)
test_kb.add_fact(fact_2)
test_kb.add_rule(rule_1)
test_kb.add_rule(rule_2)

print("=" * 65)
print("SHARED LOGICAL REPRESENTATION — TEST")
print("=" * 65)

print("\nKnowledge-base summary:")
for key, value in test_kb.summary().items():
    print(f"{key}:")
    print(value)

print("\nLiteral test:")
print("Positive:", make_literal("The river is fast"))
print("Negative:", make_literal("not The river is fast"))

assert len(test_kb.facts) == 2
assert len(test_kb.rules) == 2
assert test_kb.contains(fact_1)

print("\nAll representation tests passed.")
print("Section 4 completed successfully.")


SHARED LOGICAL REPRESENTATION — TEST

Knowledge-base summary:
number_of_facts:
2
number_of_rules:
2
facts:
['the river is fast', 'the wall dances gracefully']
rules:
['the river is fast -> the river becomes famous', 'the river becomes famous AND the wall dances gracefully -> the river is interesting']

Literal test:
Positive: the river is fast
Negative: not the river is fast

All representation tests passed.
Section 4 completed successfully.


In [5]:

# ============================================================
# SECTION 5 — RULETAKER NATURAL-LANGUAGE PARSER
# ============================================================

import re
from collections import Counter

# ------------------------------------------------------------
# 1. Clean RuleTaker sentences
# ------------------------------------------------------------

def clean_ruletaker_sentence(text):
    """Normalize one RuleTaker fact or rule sentence."""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[.!?]+$", "", text)
    return text.strip()


# ------------------------------------------------------------
# 2. Parse a single implication rule
# ------------------------------------------------------------

RULE_PATTERNS = [
    (
        "if_then",
        re.compile(
            r"^if\s+(.+?),\s*then\s+(.+)$",
            re.IGNORECASE
        )
    ),
    (
        "assuming",
        re.compile(
            r"^assuming\s+(.+?),\s*we can say\s+(.+)$",
            re.IGNORECASE
        )
    ),
    (
        "based_on",
        re.compile(
            r"^based on\s+(.+?),\s*it is evident that\s+(.+)$",
            re.IGNORECASE
        )
    ),
    (
        "given_that",
        re.compile(
            r"^given that\s+(.+?),\s*it follows that\s+(.+)$",
            re.IGNORECASE
        )
    ),
    (
        "when",
        re.compile(
            r"^when\s+(.+?),\s*we can conclude\s+(.+)$",
            re.IGNORECASE
        )
    )
]


def parse_ruletaker_rule(line, rule_id=None):
    """
    Parse a RuleTaker implication into a LogicalRule.

    Returns:
        (LogicalRule, pattern_name) if parsing succeeds
        (None, None) otherwise
    """

    sentence = clean_ruletaker_sentence(line)

    for pattern_name, pattern in RULE_PATTERNS:
        match = pattern.match(sentence)

        if match:
            antecedent_text = clean_ruletaker_sentence(match.group(1))
            consequent_text = clean_ruletaker_sentence(match.group(2))

            if not antecedent_text or not consequent_text:
                return None, None

            rule = LogicalRule(
                antecedents=(make_literal(antecedent_text),),
                consequent=make_literal(consequent_text),
                rule_id=rule_id
            )

            return rule, pattern_name

    return None, None


# ------------------------------------------------------------
# 3. Parse facts and rules from one context
# ------------------------------------------------------------

def parse_ruletaker_context(context):
    """
    Parse a RuleTaker context into a KnowledgeBase.

    Expected format:
        Facts:
        fact 1.
        fact 2.
        Rules:
        If condition, then conclusion.
    """

    kb = KnowledgeBase()

    current_section = None
    fact_count = 0
    rule_count = 0

    parsed_facts = []
    parsed_rules = []
    unparsed_lines = []
    pattern_counts = Counter()

    lines = str(context).splitlines()

    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()

        if not line:
            continue

        section_header = line.lower().rstrip(":")

        if section_header == "facts":
            current_section = "facts"
            continue

        if section_header == "rules":
            current_section = "rules"
            continue

        if current_section == "facts":
            fact_text = clean_ruletaker_sentence(line)

            if fact_text:
                literal = make_literal(fact_text)
                kb.add_fact(literal)
                parsed_facts.append(fact_text)
                fact_count += 1
            else:
                unparsed_lines.append({
                    "line_number": line_number,
                    "text": raw_line,
                    "section": current_section
                })

        elif current_section == "rules":
            rule_id = f"R{rule_count + 1}"

            rule, pattern_name = parse_ruletaker_rule(
                line,
                rule_id=rule_id
            )

            if rule is not None:
                kb.add_rule(rule)
                parsed_rules.append(str(rule))
                pattern_counts[pattern_name] += 1
                rule_count += 1
            else:
                unparsed_lines.append({
                    "line_number": line_number,
                    "text": raw_line,
                    "section": current_section
                })

        else:
            unparsed_lines.append({
                "line_number": line_number,
                "text": raw_line,
                "section": "unknown"
            })

    result = {
        "knowledge_base": kb,
        "fact_count": fact_count,
        "rule_count": rule_count,
        "parsed_facts": parsed_facts,
        "parsed_rules": parsed_rules,
        "pattern_counts": dict(pattern_counts),
        "unparsed_lines": unparsed_lines
    }

    return result


# ------------------------------------------------------------
# 4. Test on the actual RuleTaker training example
# ------------------------------------------------------------

sample_context = RULETAKER_DF["train"].iloc[0]["context"]

sample_result = parse_ruletaker_context(sample_context)
sample_kb = sample_result["knowledge_base"]

print("=" * 70)
print("RULETAKER PARSER — SAMPLE TEST")
print("=" * 70)

print("\nExtracted facts:")
for fact in sorted(sample_kb.facts, key=str):
    print(" -", fact)

print("\nExtracted rules:")
for rule in sample_kb.rules:
    print(" -", rule)

print("\nParser statistics:")
print("Facts:", sample_result["fact_count"])
print("Rules:", sample_result["rule_count"])
print("Rule patterns:", sample_result["pattern_counts"])
print("Unparsed lines:", len(sample_result["unparsed_lines"]))

if sample_result["unparsed_lines"]:
    print("\nUnparsed line examples:")
    for item in sample_result["unparsed_lines"][:5]:
        print(item)


# ------------------------------------------------------------
# 5. Measure parser coverage across all RuleTaker splits
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RULETAKER PARSER COVERAGE")
print("=" * 70)

RULETAKER_PARSE_STATS = {}

for split_name, df in RULETAKER_DF.items():

    total_facts = 0
    total_rules = 0
    total_unparsed = 0
    total_lines = 0
    pattern_totals = Counter()

    for context in df["context"]:
        result = parse_ruletaker_context(context)

        total_facts += result["fact_count"]
        total_rules += result["rule_count"]
        total_unparsed += len(result["unparsed_lines"])
        total_lines += (
            result["fact_count"]
            + result["rule_count"]
            + len(result["unparsed_lines"])
        )

        pattern_totals.update(result["pattern_counts"])

    coverage = (
        (total_facts + total_rules) / total_lines
        if total_lines else 0.0
    )

    RULETAKER_PARSE_STATS[split_name] = {
        "examples": len(df),
        "facts": total_facts,
        "rules": total_rules,
        "unparsed_lines": total_unparsed,
        "total_lines": total_lines,
        "coverage": coverage,
        "pattern_counts": dict(pattern_totals)
    }

    print(f"\n{split_name.upper()}")
    print("Examples:", len(df))
    print("Facts:", total_facts)
    print("Rules:", total_rules)
    print("Unparsed lines:", total_unparsed)
    print(f"Parser coverage: {coverage:.4%}")
    print("Pattern distribution:", dict(pattern_totals))


# ------------------------------------------------------------
# 6. Save parser statistics
# ------------------------------------------------------------

parser_stats_path = OUTPUT_DIR / "ruletaker_parser_stats.json"

with open(parser_stats_path, "w", encoding="utf-8") as file:
    json.dump(RULETAKER_PARSE_STATS, file, indent=4)

print("\nParser statistics saved to:", parser_stats_path)
print("\nSection 5 completed successfully.")


RULETAKER PARSER — SAMPLE TEST

Extracted facts:
 - the river becomes famous
 - the river dances gracefully
 - the wall dances gracefully

Extracted rules:
 - the river dances gracefully -> the river is fast
 - the wall dances gracefully -> the wall dances gracefully
 - the river becomes famous -> the wall dances gracefully
 - the river becomes famous -> the river becomes famous
 - the sandwich dances gracefully -> the river is fast
 - the river is fast -> the river is fast

Parser statistics:
Facts: 3
Rules: 6
Rule patterns: {'assuming': 3, 'if_then': 1, 'based_on': 1, 'given_that': 1}
Unparsed lines: 0

RULETAKER PARSER COVERAGE

TRAIN
Examples: 1000
Facts: 3000
Rules: 7267
Unparsed lines: 0
Parser coverage: 100.0000%
Pattern distribution: {'assuming': 1470, 'if_then': 1448, 'based_on': 1429, 'given_that': 1496, 'when': 1424}

TEST
Examples: 250
Facts: 750
Rules: 1841
Unparsed lines: 0
Parser coverage: 100.0000%
Pattern distribution: {'when': 349, 'given_that': 385, 'if_then': 360, '

In [6]:

# ============================================================
# SECTION 6 — RULETAKER CLASSICAL REASONING ENGINE
# ============================================================

from collections import defaultdict, deque
from time import perf_counter

# ------------------------------------------------------------
# 1. Forward chaining
# ------------------------------------------------------------

class ForwardChainingEngine:
    """
    Propositional forward-chaining engine.

    Starts with known facts and repeatedly applies Horn rules
    until no additional conclusions can be derived.
    """

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

        self.derived_facts = set()
        self.proofs = {}
        self.iterations = 0

    def run(self):
        self.derived_facts = set(self.kb.facts)
        self.proofs = {
            fact: {
                "type": "given_fact",
                "rule_id": None,
                "premises": ()
            }
            for fact in self.kb.facts
        }

        # Index rules by each antecedent for efficient activation.
        rules_by_premise = defaultdict(list)

        for rule_index, rule in enumerate(self.kb.rules):
            for premise in rule.antecedents:
                rules_by_premise[premise].append(rule_index)

        # Track how many antecedents of each rule remain unproved.
        remaining = [
            len(rule.antecedents)
            for rule in self.kb.rules
        ]

        # Empty-premise rules can be activated immediately.
        agenda = deque(self.derived_facts)

        activated_rules = set()

        for rule_index, rule in enumerate(self.kb.rules):
            if remaining[rule_index] == 0:
                activated_rules.add(rule_index)
                agenda.append(rule.consequent)

        # Track which premises have already satisfied each rule.
        satisfied = [set() for _ in self.kb.rules]

        while agenda:
            known_literal = agenda.popleft()
            self.iterations += 1

            for rule_index in rules_by_premise.get(known_literal, []):

                if rule_index in activated_rules:
                    continue

                if known_literal in satisfied[rule_index]:
                    continue

                satisfied[rule_index].add(known_literal)
                remaining[rule_index] -= 1

                if remaining[rule_index] == 0:
                    activated_rules.add(rule_index)

                    rule = self.kb.rules[rule_index]
                    conclusion = rule.consequent

                    if conclusion not in self.derived_facts:
                        self.derived_facts.add(conclusion)

                        self.proofs[conclusion] = {
                            "type": "rule_application",
                            "rule_id": rule.rule_id,
                            "premises": rule.antecedents
                        }

                        agenda.append(conclusion)

        return self

    def entails(self, query):
        return query in self.derived_facts

    def get_proof(self, query):
        return self.proofs.get(query)

    def get_trace(self, query):
        """
        Return a recursive derivation trace for a query.
        """
        visited = set()
        trace = []

        def explain(literal, depth=0):
            if literal in visited:
                return

            visited.add(literal)

            proof = self.proofs.get(literal)

            if proof is None:
                return

            indent = "  " * depth

            if proof["type"] == "given_fact":
                trace.append(
                    f"{indent}FACT: {literal}"
                )
                return

            trace.append(
                f"{indent}APPLY {proof['rule_id']}: {literal}"
            )

            for premise in proof["premises"]:
                explain(premise, depth + 1)

        explain(query)

        return trace


# ------------------------------------------------------------
# 2. Backward chaining
# ------------------------------------------------------------

class BackwardChainingEngine:
    """
    Goal-directed backward chaining.

    Attempts to prove a query by finding rules whose consequent
    matches the query and recursively proving their antecedents.
    """

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

        self.rules_by_conclusion = defaultdict(list)

        for rule in self.kb.rules:
            self.rules_by_conclusion[rule.consequent].append(rule)

        self.memo = {}
        self.active_goals = set()
        self.proofs = {}
        self.steps = 0

    def prove(self, query):
        """
        Return True if the query can be established from the KB.
        """

        self.memo = {}
        self.active_goals = set()
        self.proofs = {}
        self.steps = 0

        return self._prove(query)

    def _prove(self, goal):
        self.steps += 1

        # Directly known fact.
        if goal in self.kb.facts:
            self.proofs[goal] = {
                "type": "given_fact",
                "rule_id": None,
                "premises": ()
            }
            return True

        # Previously evaluated goal.
        if goal in self.memo:
            return self.memo[goal]

        # Avoid infinite recursion from cyclic rules.
        if goal in self.active_goals:
            return False

        self.active_goals.add(goal)

        for rule in self.rules_by_conclusion.get(goal, []):

            premises_proved = True

            for premise in rule.antecedents:
                if not self._prove(premise):
                    premises_proved = False
                    break

            if premises_proved:
                self.proofs[goal] = {
                    "type": "rule_application",
                    "rule_id": rule.rule_id,
                    "premises": rule.antecedents
                }

                self.active_goals.remove(goal)
                self.memo[goal] = True
                return True

        self.active_goals.remove(goal)
        self.memo[goal] = False

        return False

    def get_trace(self, query):
        """
        Return the successful backward proof trace.
        """
        visited = set()
        trace = []

        def explain(literal, depth=0):
            if literal in visited:
                return

            visited.add(literal)

            proof = self.proofs.get(literal)

            if proof is None:
                return

            indent = "  " * depth

            if proof["type"] == "given_fact":
                trace.append(f"{indent}FACT: {literal}")
                return

            trace.append(
                f"{indent}APPLY {proof['rule_id']}: {literal}"
            )

            for premise in proof["premises"]:
                explain(premise, depth + 1)

        explain(query)

        return trace


# ------------------------------------------------------------
# 3. Unified RuleTaker prediction interface
# ------------------------------------------------------------

def predict_ruletaker(context, statement, method="forward"):
    """
    Parse a RuleTaker example and predict whether its statement
    is entailed by the knowledge base.

    method:
        "forward"  -> forward chaining
        "backward" -> backward chaining

    Returns prediction, timing, and proof trace.
    """

    start_time = perf_counter()

    parsed = parse_ruletaker_context(context)
    kb = parsed["knowledge_base"]

    query = make_literal(statement)

    if method == "forward":
        engine = ForwardChainingEngine(kb).run()
        prediction = engine.entails(query)
        trace = engine.get_trace(query)
        steps = engine.iterations

    elif method == "backward":
        engine = BackwardChainingEngine(kb)
        prediction = engine.prove(query)
        trace = engine.get_trace(query)
        steps = engine.steps

    else:
        raise ValueError(
            "method must be 'forward' or 'backward'"
        )

    elapsed = perf_counter() - start_time

    return {
        "prediction": prediction,
        "query": str(query),
        "method": method,
        "inference_time": elapsed,
        "reasoning_steps": steps,
        "trace": trace,
        "facts_derived": (
            len(engine.derived_facts)
            if method == "forward"
            else None
        )
    }


# ------------------------------------------------------------
# 4. Test on the actual RuleTaker sample
# ------------------------------------------------------------

sample_row = RULETAKER_DF["train"].iloc[0]

sample_context = sample_row["context"]
sample_statement = sample_row["statement"]
sample_label = sample_row["flag"] == "True"

print("=" * 70)
print("RULETAKER REASONING ENGINE — SAMPLE TEST")
print("=" * 70)

print("\nQuery:", sample_statement)
print("Expected label:", sample_label)

forward_result = predict_ruletaker(
    sample_context,
    sample_statement,
    method="forward"
)

backward_result = predict_ruletaker(
    sample_context,
    sample_statement,
    method="backward"
)

for method_name, result in [
    ("FORWARD CHAINING", forward_result),
    ("BACKWARD CHAINING", backward_result)
]:
    print(f"\n{method_name}")
    print("Prediction:", result["prediction"])
    print("Inference time:", f"{result['inference_time']:.8f} seconds")
    print("Reasoning steps:", result["reasoning_steps"])

    if result["trace"]:
        print("Proof trace:")
        for line in result["trace"]:
            print(" ", line)
    else:
        print("No proof found.")

    print(
        "Matches dataset label:",
        result["prediction"] == sample_label
    )


# ------------------------------------------------------------
# 5. Unit tests for inference behaviour
# ------------------------------------------------------------

test_kb = KnowledgeBase()

a = make_literal("a")
b = make_literal("b")
c = make_literal("c")
d = make_literal("d")
unknown = make_literal("unknown")

test_kb.add_fact(a)

test_kb.add_rule(
    LogicalRule((a,), b, "T1")
)

test_kb.add_rule(
    LogicalRule((b,), c, "T2")
)

test_kb.add_rule(
    LogicalRule((c, d), unknown, "T3")
)

# Forward chaining
fc_test = ForwardChainingEngine(test_kb).run()

assert fc_test.entails(a)
assert fc_test.entails(b)
assert fc_test.entails(c)
assert not fc_test.entails(unknown)

# Backward chaining
bc_test = BackwardChainingEngine(test_kb)

assert bc_test.prove(a)
assert bc_test.prove(b)
assert bc_test.prove(c)
assert not bc_test.prove(unknown)

# Test cyclic rules without infinite recursion
cyclic_kb = KnowledgeBase()

cyclic_kb.add_rule(
    LogicalRule(
        (make_literal("cycle_b"),),
        make_literal("cycle_a"),
        "C1"
    )
)

cyclic_kb.add_rule(
    LogicalRule(
        (make_literal("cycle_a"),),
        make_literal("cycle_b"),
        "C2"
    )
)

cycle_engine = BackwardChainingEngine(cyclic_kb)

assert not cycle_engine.prove(make_literal("cycle_a"))

print("\n" + "=" * 70)
print("UNIT TEST RESULTS")
print("=" * 70)
print("Forward chaining tests: PASSED")
print("Backward chaining tests: PASSED")
print("Multi-step inference test: PASSED")
print("Unknown query test: PASSED")
print("Cyclic-rule termination test: PASSED")

print("\nSection 6 completed successfully.")


RULETAKER REASONING ENGINE — SAMPLE TEST

Query: The sandwich dances gracefully.
Expected label: False

FORWARD CHAINING
Prediction: False
Inference time: 0.00050026 seconds
Reasoning steps: 4
No proof found.
Matches dataset label: True

BACKWARD CHAINING
Prediction: False
Inference time: 0.00036547 seconds
Reasoning steps: 1
No proof found.
Matches dataset label: True

UNIT TEST RESULTS
Forward chaining tests: PASSED
Backward chaining tests: PASSED
Multi-step inference test: PASSED
Unknown query test: PASSED
Cyclic-rule termination test: PASSED

Section 6 completed successfully.


In [7]:

# ============================================================
# SECTION 7 — RULETAKER EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import json
from time import perf_counter
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Evaluate one RuleTaker split with one inference method
# ------------------------------------------------------------

def evaluate_ruletaker_split(df, split_name, method):
    """
    Evaluate a reasoning method on a complete RuleTaker split.
    """

    predictions = []
    actual_labels = []
    inference_times = []
    reasoning_steps = []

    total_start = perf_counter()

    for row in tqdm(
        df.itertuples(index=False),
        total=len(df),
        desc=f"{split_name} | {method}"
    ):
        result = predict_ruletaker(
            context=row.context,
            statement=row.statement,
            method=method
        )

        predictions.append(result["prediction"])
        actual_labels.append(row.flag == "True")
        inference_times.append(result["inference_time"])
        reasoning_steps.append(result["reasoning_steps"])

    total_elapsed = perf_counter() - total_start

    y_true = np.asarray(actual_labels, dtype=bool)
    y_pred = np.asarray(predictions, dtype=bool)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[False, True]
    )

    metrics = {
        "dataset": split_name,
        "method": method,
        "examples": len(df),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "total_elapsed_seconds": total_elapsed,
        "total_inference_seconds": float(sum(inference_times)),
        "average_inference_seconds": float(np.mean(inference_times)),
        "median_inference_seconds": float(np.median(inference_times)),
        "throughput_examples_per_second": (
            len(df) / total_elapsed if total_elapsed else 0.0
        ),
        "average_reasoning_steps": float(np.mean(reasoning_steps)),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1])
    }

    return metrics, {
        "actual": y_true.tolist(),
        "predicted": y_pred.tolist(),
        "inference_times": inference_times,
        "reasoning_steps": reasoning_steps
    }


# ------------------------------------------------------------
# 2. Run both methods on all RuleTaker splits
# ------------------------------------------------------------

RULETAKER_RESULTS = []
RULETAKER_PREDICTIONS = {}

for split_name in ["train", "test", "ood_test"]:

    df = RULETAKER_DF[split_name]

    for method in ["forward", "backward"]:

        metrics, predictions = evaluate_ruletaker_split(
            df=df,
            split_name=split_name,
            method=method
        )

        RULETAKER_RESULTS.append(metrics)

        RULETAKER_PREDICTIONS[
            (split_name, method)
        ] = predictions


# ------------------------------------------------------------
# 3. Display evaluation table
# ------------------------------------------------------------

results_df = pd.DataFrame(RULETAKER_RESULTS)

display_columns = [
    "dataset",
    "method",
    "examples",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "average_inference_seconds",
    "throughput_examples_per_second",
    "average_reasoning_steps"
]

print("\n" + "=" * 100)
print("RULETAKER — PERFORMANCE COMPARISON")
print("=" * 100)

display(
    results_df[display_columns].style.format({
        "accuracy": "{:.4%}",
        "precision": "{:.4%}",
        "recall": "{:.4%}",
        "f1": "{:.4%}",
        "average_inference_seconds": "{:.8f}",
        "throughput_examples_per_second": "{:.2f}",
        "average_reasoning_steps": "{:.2f}"
    })
)


# ------------------------------------------------------------
# 4. Print confusion matrices and detailed reports
# ------------------------------------------------------------

for split_name in ["train", "test", "ood_test"]:

    for method in ["forward", "backward"]:

        predictions = RULETAKER_PREDICTIONS[
            (split_name, method)
        ]

        y_true = predictions["actual"]
        y_pred = predictions["predicted"]

        print("\n" + "-" * 70)
        print(f"{split_name.upper()} — {method.upper()}")

        print("\nConfusion matrix:")
        print("Rows = actual [False, True]")
        print("Columns = predicted [False, True]")
        print(
            confusion_matrix(
                y_true,
                y_pred,
                labels=[False, True]
            )
        )

        print("\nClassification report:")
        print(
            classification_report(
                y_true,
                y_pred,
                labels=[False, True],
                target_names=["False", "True"],
                zero_division=0
            )
        )


# ------------------------------------------------------------
# 5. Save results and predictions
# ------------------------------------------------------------

results_path = OUTPUT_DIR / "ruletaker_evaluation.csv"

results_df.to_csv(results_path, index=False)

predictions_path = OUTPUT_DIR / "ruletaker_predictions.json"

serializable_predictions = {}

for (split_name, method), values in RULETAKER_PREDICTIONS.items():
    key = f"{split_name}_{method}"

    serializable_predictions[key] = values

with open(predictions_path, "w", encoding="utf-8") as file:
    json.dump(serializable_predictions, file, indent=2)

# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RULETAKER EVALUATION COMPLETED")
print("=" * 100)

print("Evaluation records:", len(results_df))
print("Methods:", results_df["method"].unique().tolist())
print("Splits:", results_df["dataset"].unique().tolist())

print("\nResults saved:")
print(results_path)
print(predictions_path)

print("\nSection 7 completed successfully.")


train | forward:   0%|          | 0/1000 [00:00<?, ?it/s]

train | backward:   0%|          | 0/1000 [00:00<?, ?it/s]

test | forward:   0%|          | 0/250 [00:00<?, ?it/s]

test | backward:   0%|          | 0/250 [00:00<?, ?it/s]

ood_test | forward:   0%|          | 0/375 [00:00<?, ?it/s]

ood_test | backward:   0%|          | 0/375 [00:00<?, ?it/s]


RULETAKER — PERFORMANCE COMPARISON


,dataset,method,examples,accuracy,precision,recall,f1,average_inference_seconds,throughput_examples_per_second,average_reasoning_steps
0,train,forward,1000,98.3000%,97.2177%,100.0000%,98.5892%,0.00045053,2109.65,6.35
1,train,backward,1000,98.3000%,97.2177%,100.0000%,98.5892%,0.00041395,2288.06,1.98
2,test,forward,250,97.6000%,96.2963%,100.0000%,98.1132%,0.00047262,1887.29,6.52
3,test,backward,250,97.6000%,96.2963%,100.0000%,98.1132%,0.00043033,2059.74,2.02
4,ood_test,forward,375,99.4667%,98.4962%,100.0000%,99.2424%,0.00042176,2157.60,6.89
5,ood_test,backward,375,99.4667%,98.4962%,100.0000%,99.2424%,0.00040014,2265.32,1.72



----------------------------------------------------------------------
TRAIN — FORWARD

Confusion matrix:
Rows = actual [False, True]
Columns = predicted [False, True]
[[389  17]
 [  0 594]]

Classification report:
              precision    recall  f1-score   support

       False       1.00      0.96      0.98       406
        True       0.97      1.00      0.99       594

    accuracy                           0.98      1000
   macro avg       0.99      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000


----------------------------------------------------------------------
TRAIN — BACKWARD

Confusion matrix:
Rows = actual [False, True]
Columns = predicted [False, True]
[[389  17]
 [  0 594]]

Classification report:
              precision    recall  f1-score   support

       False       1.00      0.96      0.98       406
        True       0.97      1.00      0.99       594

    accuracy                           0.98      1000
   macro avg       0.9

In [8]:

# ============================================================
# SECTION 8 — COUNTERBENCH CAUSAL GRAPH REPRESENTATION
# ============================================================

from dataclasses import dataclass, field
from collections import defaultdict, Counter
import re
import json

# ------------------------------------------------------------
# 1. Causal edge representation
# ------------------------------------------------------------

@dataclass(frozen=True)
class CausalEdge:
    """
    A causal relationship.

    parents:
        One or more causal parent variables.

    child:
        The affected variable.

    A single parent represents:
        X -> Y

    Two parents represent:
        X OR Y -> Z
    """
    parents: tuple
    child: str

    def __post_init__(self):
        object.__setattr__(
            self,
            "parents",
            tuple(normalize_text(x) for x in self.parents)
        )
        object.__setattr__(
            self,
            "child",
            normalize_text(self.child)
        )

    def __str__(self):
        return f"{' OR '.join(self.parents)} -> {self.child}"


# ------------------------------------------------------------
# 2. Causal graph
# ------------------------------------------------------------

@dataclass
class CausalGraph:
    edges: list = field(default_factory=list)

    def add_edge(self, edge):
        if not isinstance(edge, CausalEdge):
            raise TypeError("Expected a CausalEdge.")
        self.edges.append(edge)

    @property
    def variables(self):
        result = set()

        for edge in self.edges:
            result.add(edge.child)
            result.update(edge.parents)

        return result

    def parents_of(self, variable):
        variable = normalize_text(variable)

        return [
            edge.parents
            for edge in self.edges
            if edge.child == variable
        ]

    def children_of(self, variable):
        variable = normalize_text(variable)

        return [
            edge.child
            for edge in self.edges
            if variable in edge.parents
        ]

    def summary(self):
        return {
            "variables": sorted(self.variables),
            "number_of_variables": len(self.variables),
            "number_of_edges": len(self.edges),
            "edges": [str(edge) for edge in self.edges]
        }


# ------------------------------------------------------------
# 3. Parse causal statements from CounterBench text
# ------------------------------------------------------------

# CounterBench uses generated variable names such as:
# Ziklo, Blaf, Trune, Vork, Lumbo.
#
# Supported patterns:
#   X causes Y
#   X or Y causes Z

SINGLE_CAUSE_PATTERN = re.compile(
    r"\b([A-Za-z][A-Za-z0-9_-]*)\s+causes\s+"
    r"([A-Za-z][A-Za-z0-9_-]*)\b",
    re.IGNORECASE
)

DISJUNCTIVE_CAUSE_PATTERN = re.compile(
    r"\b([A-Za-z][A-Za-z0-9_-]*)\s+or\s+"
    r"([A-Za-z][A-Za-z0-9_-]*)\s+causes\s+"
    r"([A-Za-z][A-Za-z0-9_-]*)\b",
    re.IGNORECASE
)


def parse_causal_graph(given_info):
    """
    Extract causal edges from CounterBench given_info.

    Returns:
        graph: CausalGraph
        diagnostics: parsing information

    Disjunctive relationships are stored as one edge with
    multiple parents, not as two independent edges.
    """

    text = str(given_info)
    graph = CausalGraph()

    # First identify disjunctive relationships.
    disjunctive_matches = list(
        DISJUNCTIVE_CAUSE_PATTERN.finditer(text)
    )

    occupied_spans = [
        match.span()
        for match in disjunctive_matches
    ]

    for match in disjunctive_matches:
        parent_1, parent_2, child = match.groups()

        graph.add_edge(
            CausalEdge(
                parents=(parent_1, parent_2),
                child=child
            )
        )

    # Then identify ordinary single-parent relationships.
    single_matches = list(
        SINGLE_CAUSE_PATTERN.finditer(text)
    )

    single_count = 0

    for match in single_matches:
        # Do not parse a disjunctive relationship again.
        if any(
            start <= match.start() and match.end() <= end
            for start, end in occupied_spans
        ):
            continue

        parent, child = match.groups()

        graph.add_edge(
            CausalEdge(
                parents=(parent,),
                child=child
            )
        )

        single_count += 1

    diagnostics = {
        "single_parent_edges": single_count,
        "disjunctive_edges": len(disjunctive_matches),
        "total_edges": len(graph.edges),
        "variables": len(graph.variables)
    }

    return graph, diagnostics


# ------------------------------------------------------------
# 4. Parse one Alpha example
# ------------------------------------------------------------

alpha_sample = COUNTERBENCH_ALPHA.iloc[0]

alpha_graph, alpha_diagnostics = parse_causal_graph(
    alpha_sample["given_info"]
)

print("=" * 70)
print("COUNTERBENCH — SAMPLE CAUSAL GRAPH")
print("=" * 70)

print("\nQuestion:", alpha_sample["question"])
print("Given information:", alpha_sample["given_info"])
print("Expected answer:", alpha_sample["answer"])

print("\nExtracted edges:")
for edge in alpha_graph.edges:
    print(" ", edge)

print("\nGraph summary:")
print(json.dumps(alpha_graph.summary(), indent=2))

print("\nParsing diagnostics:")
print(alpha_diagnostics)


# ------------------------------------------------------------
# 5. Parse one Backdoor example
# ------------------------------------------------------------

backdoor_sample = COUNTERBENCH_BACKDOOR.iloc[0]

backdoor_graph, backdoor_diagnostics = parse_causal_graph(
    backdoor_sample["given_info"]
)

print("\n" + "=" * 70)
print("COUNTERBENCH — BACKDOOR SAMPLE")
print("=" * 70)

print("\nQuestion:", backdoor_sample["question"])
print("Given information:", backdoor_sample["given_info"])

print("\nExtracted edges:")
for edge in backdoor_graph.edges:
    print(" ", edge)

print("\nParsing diagnostics:")
print(backdoor_diagnostics)


# ------------------------------------------------------------
# 6. Measure parsing coverage across both datasets
# ------------------------------------------------------------

COUNTERBENCH_PARSE_STATS = {}

for dataset_name, df in {
    "alpha": COUNTERBENCH_ALPHA,
    "backdoor": COUNTERBENCH_BACKDOOR
}.items():

    total_examples = len(df)
    examples_with_edges = 0
    total_edges = 0
    total_variables = 0
    single_edges = 0
    disjunctive_edges = 0
    empty_graph_examples = []

    for index, row in df.iterrows():

        graph, diagnostics = parse_causal_graph(
            row["given_info"]
        )

        if graph.edges:
            examples_with_edges += 1
        else:
            empty_graph_examples.append(str(row["question_id"]))

        total_edges += diagnostics["total_edges"]
        total_variables += diagnostics["variables"]
        single_edges += diagnostics["single_parent_edges"]
        disjunctive_edges += diagnostics["disjunctive_edges"]

    stats = {
        "examples": total_examples,
        "examples_with_edges": examples_with_edges,
        "empty_graph_examples": len(empty_graph_examples),
        "total_edges": total_edges,
        "single_parent_edges": single_edges,
        "disjunctive_edges": disjunctive_edges,
        "total_variable_occurrences": total_variables,
        "graph_extraction_coverage": (
            examples_with_edges / total_examples
            if total_examples else 0.0
        ),
        "empty_graph_question_ids": empty_graph_examples[:20]
    }

    COUNTERBENCH_PARSE_STATS[dataset_name] = stats

    print("\n" + "-" * 70)
    print(dataset_name.upper())
    print("-" * 70)

    for key, value in stats.items():
        print(f"{key}: {value}")


# ------------------------------------------------------------
# 7. Verify graph representation behaviour
# ------------------------------------------------------------

test_graph = CausalGraph()

test_graph.add_edge(
    CausalEdge(("x",), "y")
)

test_graph.add_edge(
    CausalEdge(("y", "z"), "w")
)

assert "x" in test_graph.variables
assert "w" in test_graph.variables
assert test_graph.parents_of("y") == [("x",)]
assert test_graph.children_of("y") == ["w"]

assert len(alpha_graph.edges) == alpha_diagnostics["total_edges"]
assert len(backdoor_graph.edges) == backdoor_diagnostics["total_edges"]


# ------------------------------------------------------------
# 8. Save parsing statistics
# ------------------------------------------------------------

counterbench_stats_path = (
    OUTPUT_DIR / "counterbench_graph_parser_stats.json"
)

with open(
    counterbench_stats_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        COUNTERBENCH_PARSE_STATS,
        file,
        indent=4
    )

print("\n" + "=" * 70)
print("COUNTERBENCH GRAPH REPRESENTATION TESTS")
print("=" * 70)

print("Parent lookup test: PASSED")
print("Child lookup test: PASSED")
print("Disjunctive edge representation: PASSED")
print("Graph extraction consistency: PASSED")

print("\nStatistics saved to:", counterbench_stats_path)
print("\nSection 8 completed successfully.")


COUNTERBENCH — SAMPLE CAUSAL GRAPH

Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Given information: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
Expected answer: no

Extracted edges:
  ziklo -> blaf
  blaf -> trune
  trune -> vork
  vork -> lumbo

Graph summary:
{
  "variables": [
    "blaf",
    "lumbo",
    "trune",
    "vork",
    "ziklo"
  ],
  "number_of_variables": 5,
  "number_of_edges": 4,
  "edges": [
    "ziklo -> blaf",
    "blaf -> trune",
    "trune -> vork",
    "vork -> lumbo"
  ]
}

Parsing diagnostics:
{'single_parent_edges': 4, 'disjunctive_edges': 0, 'total_edges': 4, 'variables': 5}

COUNTERBENCH — BACKDOOR SAMPLE

Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Given information: We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune

Extracted edges:
  ziklo OR blaf -> trune
  blaf -> ziklo
  trune -> vork
  vor

In [9]:

# ============================================================
# SECTION 10: COUNTERBENCH CAUSAL MECHANISM AND QUESTION PARSER
# ============================================================

import re
import json
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

# ------------------------------------------------------------
# 10.1 Configuration
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/AI_Assignment_2026")
NORMALIZED_DIR = PROJECT_DIR / "data" / "normalized"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIABLE = r"[A-Za-z][A-Za-z0-9_-]*"

STOP_WORDS = {
    "if", "and", "or", "instead", "of", "would", "occur",
    "assume", "assumes", "suppose", "supposes", "based",
    "on", "this", "assumption", "further", "then", "not"
}


# ------------------------------------------------------------
# 10.2 Causal mechanism representation
# ------------------------------------------------------------

@dataclass(frozen=True)
class CausalMechanism:
    parents: tuple
    parent_operator: str
    child: str
    child_value: bool = True
    parent_values: tuple = ()

    def __post_init__(self):
        object.__setattr__(self, "parents", tuple(normalize_variable(x) for x in self.parents))
        object.__setattr__(self, "child", normalize_variable(self.child))
        object.__setattr__(self, "parent_operator", self.parent_operator.upper())
        values = self.parent_values or tuple(True for _ in self.parents)
        object.__setattr__(self, "parent_values", tuple(bool(v) for v in values))
        if self.parent_operator not in {"AND", "OR"}:
            raise ValueError("parent_operator must be AND or OR")
        if not self.parents:
            raise ValueError("A causal mechanism needs at least one parent")
        if len(self.parents) != len(self.parent_values):
            raise ValueError("parent_values must match parents length")


# ------------------------------------------------------------
# 10.3 Causal statement patterns
# ------------------------------------------------------------

COMBINATION_CAUSE_PATTERN = re.compile(
    rf"^\s*({VARIABLE})\s+(and|or)\s+({VARIABLE})"
    rf"(?:\s+together)?\s+causes?\s+(not\s+)?({VARIABLE})\s*$",
    re.IGNORECASE
)

MULTIPLE_EFFECT_PATTERN = re.compile(
    rf"^\s*({VARIABLE})\s+causes?\s+(.+?)\s*$",
    re.IGNORECASE
)


# ------------------------------------------------------------
# 10.4 Helpers for causal statement parsing
# ------------------------------------------------------------

def normalize_variable(name):
    return name.strip().lower()


def parse_effect_list(effect_text):
    """
    Parse effects such as:
        Blaf
        not Blaf
        Blaf and Vork
        Blaf, Trune and Vork
    """

    parts = re.split(
        r"\s*,\s*|\s+and\s+",
        effect_text.strip(),
        flags=re.IGNORECASE
    )

    effects = []

    for part in parts:
        part = part.strip().strip(".;")

        match = re.fullmatch(
            rf"(not\s+)?({VARIABLE})",
            part,
            re.IGNORECASE
        )

        if match:
            negation, variable = match.groups()

            effects.append(
                (
                    normalize_variable(variable),
                    not bool(negation)
                )
            )

    return effects


# ------------------------------------------------------------
# 10.5 Parse causal mechanisms
# ------------------------------------------------------------

def parse_causal_mechanisms(given_info):
    """
    Supported patterns:
      A causes B
      A causes not B
      A and B together cause C
      A and B cause C
      A or B causes C
      A causes B and C
      A causes B, C and D
    """

    mechanisms = []

    if not isinstance(given_info, str):
        return mechanisms

    statements = re.split(r"[\n;]+", given_info)

    for statement in statements:
        statement = statement.strip().strip(".")

        if not statement:
            continue

        # Multiple antecedents: AND / OR
        match = COMBINATION_CAUSE_PATTERN.match(statement)

        if match:
            parent1, operator, parent2, negation, child = match.groups()

            mechanisms.append(
                CausalMechanism(
                    parents=(
                        normalize_variable(parent1),
                        normalize_variable(parent2)
                    ),
                    parent_operator=operator.upper(),
                    child=normalize_variable(child),
                    child_value=not bool(negation)
                )
            )
            continue

        # Single antecedent with one or multiple effects
        match = MULTIPLE_EFFECT_PATTERN.match(statement)

        if match:
            parent, effect_text = match.groups()

            effects = parse_effect_list(effect_text)

            for child, child_value in effects:
                mechanisms.append(
                    CausalMechanism(
                        parents=(normalize_variable(parent),),
                        parent_operator="AND",
                        child=child,
                        child_value=child_value
                    )
                )

    return mechanisms


# ------------------------------------------------------------
# 10.6 Intervention representation
# ------------------------------------------------------------

@dataclass(frozen=True)
class Intervention:
    variable: str
    value: bool


@dataclass(frozen=True)
class CounterfactualQuery:
    outcome: str
    interventions: tuple
    query_type: str


# ------------------------------------------------------------
# 10.7 Intervention parser
# ------------------------------------------------------------

INTERVENTION_PATTERN = re.compile(
    rf"\b(not\s+)?({VARIABLE})\b",
    re.IGNORECASE
)


def extract_interventions(text):
    """
    Extract intervention variables and their assumed values.

    Examples:
        not Ziklo -> Ziklo = False
        not Ziklo and not Blaf -> both False
        if not Ziklo instead of Ziklo -> only Ziklo = False
    """

    if not isinstance(text, str):
        return tuple()

    # Remove baseline comparison.
    text = re.split(
        r"\binstead\s+of\b",
        text,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    # Remove leading 'if'.
    text = re.sub(
        r"^\s*if\s+",
        "",
        text,
        flags=re.IGNORECASE
    )

    interventions = []

    for match in INTERVENTION_PATTERN.finditer(text):
        negation, variable = match.groups()

        variable = normalize_variable(variable)

        if variable in STOP_WORDS:
            continue

        interventions.append(
            Intervention(
                variable=variable,
                value=not bool(negation)
            )
        )

    # Remove duplicates, preserving order.
    unique = []
    seen = set()

    for intervention in interventions:
        if intervention.variable not in seen:
            unique.append(intervention)
            seen.add(intervention.variable)

    return tuple(unique)


# ------------------------------------------------------------
# 10.8 Counterfactual question patterns
# ------------------------------------------------------------

SINGLE_QUERY_PATTERN = re.compile(
    rf"\bWould\s+({VARIABLE})\s+occur\s+if\s+(.+?)\?",
    re.IGNORECASE | re.DOTALL
)

DIRECT_QUERY_PATTERN = re.compile(
    rf"\bWould\s+({VARIABLE})\s+occur\?",
    re.IGNORECASE
)


# ------------------------------------------------------------
# 10.9 Parse counterfactual questions
# ------------------------------------------------------------

def parse_counterfactual_question(question):
    """
    Parse:
      1. Basic counterfactual questions
      2. Joint intervention questions
      3. Nested assumption questions
      4. Direct outcome questions
    """

    if not isinstance(question, str):
        return None

    question = question.strip()

    # --------------------------------------------------------
    # Case 1: Nested assumptions
    #
    # Example:
    # Assume not Ziklo, and based on this assumption,
    # further suppose not Sline. Would Lumbo occur?
    # --------------------------------------------------------

    nested_match = re.search(
        r"\bAssume\s+(.+?),\s*"
        r"and\s+based\s+on\s+this\s+assumption,\s*"
        r"further\s+suppose\s+(.+?)\.\s*"
        r"Would\s+([A-Za-z][A-Za-z0-9_-]*)\s+occur\?",
        question,
        re.IGNORECASE | re.DOTALL
    )

    if nested_match:
        first_assumption = nested_match.group(1).strip()
        second_assumption = nested_match.group(2).strip()
        outcome = nested_match.group(3)

        first_interventions = extract_interventions(first_assumption)
        second_interventions = extract_interventions(second_assumption)

        combined = list(first_interventions) + list(second_interventions)

        unique = []
        seen = set()

        for intervention in combined:
            if intervention.variable not in seen:
                unique.append(intervention)
                seen.add(intervention.variable)

        if unique:
            return CounterfactualQuery(
                outcome=normalize_variable(outcome),
                interventions=tuple(unique),
                query_type="nested"
            )

    # --------------------------------------------------------
    # Case 2: Basic or joint counterfactual
    # --------------------------------------------------------

    single_match = SINGLE_QUERY_PATTERN.search(question)

    if single_match:
        outcome, intervention_text = single_match.groups()

        interventions = extract_interventions(intervention_text)

        if interventions:
            query_type = (
                "joint" if len(interventions) > 1 else "single"
            )

            return CounterfactualQuery(
                outcome=normalize_variable(outcome),
                interventions=interventions,
                query_type=query_type
            )

    # --------------------------------------------------------
    # Case 3: Direct question
    # --------------------------------------------------------

    direct_match = DIRECT_QUERY_PATTERN.search(question)

    if direct_match:
        return CounterfactualQuery(
            outcome=normalize_variable(direct_match.group(1)),
            interventions=tuple(),
            query_type="direct"
        )

    return None


# ------------------------------------------------------------
# 10.10 Causal mechanism parser tests
# ------------------------------------------------------------

causal_test_cases = [
    (
        "Ziklo causes Blaf",
        [("ziklo", "AND", "blaf", True)]
    ),
    (
        "Ziklo causes not Blaf",
        [("ziklo", "AND", "blaf", False)]
    ),
    (
        "Blaf and Trune together cause Vork",
        [("blaf", "AND", "vork", True)]
    ),
    (
        "Trune or Vork causes Lumbo",
        [("trune", "OR", "lumbo", True)]
    ),
    (
        "Ziklo causes Blaf and Vork",
        [
            ("ziklo", "AND", "blaf", True),
            ("ziklo", "AND", "vork", True)
        ]
    ),
    (
        "Ziklo causes Blaf, Trune and Vork",
        [
            ("ziklo", "AND", "blaf", True),
            ("ziklo", "AND", "trune", True),
            ("ziklo", "AND", "vork", True)
        ]
    )
]

for statement, expected in causal_test_cases:
    parsed = parse_causal_mechanisms(statement)

    actual = [
        (
            mechanism.parents[0],
            mechanism.parent_operator,
            mechanism.child,
            mechanism.child_value
        )
        for mechanism in parsed
    ]

    assert actual == expected, (
        f"Causal parser failed for: {statement}\n"
        f"Expected: {expected}\nActual: {actual}"
    )

print("Causal mechanism parser tests: PASSED")


# ------------------------------------------------------------
# 10.11 Intervention parser tests
# ------------------------------------------------------------

intervention_tests = [
    ("not Ziklo", [("ziklo", False)]),
    (
        "not Ziklo and not Blaf",
        [("ziklo", False), ("blaf", False)]
    ),
    (
        "if not Ziklo instead of Ziklo",
        [("ziklo", False)]
    ),
    (
        "if not Ziklo and not Blaf",
        [("ziklo", False), ("blaf", False)]
    )
]

for text, expected in intervention_tests:
    parsed = extract_interventions(text)

    actual = [
        (item.variable, item.value)
        for item in parsed
    ]

    assert actual == expected, (
        f"Intervention parser failed for: {text}\n"
        f"Expected: {expected}\nActual: {actual}"
    )

print("Intervention parser tests: PASSED")


# ------------------------------------------------------------
# 10.12 Counterfactual question parser tests
# ------------------------------------------------------------

question_tests = [
    (
        "Would Lumbo occur if not Ziklo instead of Ziklo?",
        "lumbo",
        [("ziklo", False)],
        "single"
    ),
    (
        "Would Lumbo occur if not Ziklo and not Blaf?",
        "lumbo",
        [("ziklo", False), ("blaf", False)],
        "joint"
    ),
    (
        "Assume not Ziklo, and based on this assumption, "
        "further suppose not Sline. Would Lumbo occur?",
        "lumbo",
        [("ziklo", False), ("sline", False)],
        "nested"
    )
]

for question, expected_outcome, expected_interventions, expected_type in question_tests:
    parsed = parse_counterfactual_question(question)

    assert parsed is not None, (
        f"Question parser returned None: {question}"
    )

    actual_interventions = [
        (item.variable, item.value)
        for item in parsed.interventions
    ]

    assert parsed.outcome == expected_outcome, (
        f"Incorrect outcome for: {question}"
    )

    assert actual_interventions == expected_interventions, (
        f"Incorrect interventions for: {question}\n"
        f"Expected: {expected_interventions}\n"
        f"Actual: {actual_interventions}"
    )

    assert parsed.query_type == expected_type, (
        f"Incorrect query type for: {question}"
    )

print("Counterfactual question parser tests: PASSED")


# ------------------------------------------------------------
# 10.13 Load normalized CounterBench datasets
# ------------------------------------------------------------

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


alpha_path = NORMALIZED_DIR / "counterbench_alpha.jsonl"
backdoor_path = NORMALIZED_DIR / "counterbench_backdoor.jsonl"

assert alpha_path.exists(), f"Missing file: {alpha_path}"
assert backdoor_path.exists(), f"Missing file: {backdoor_path}"

alpha_records = load_jsonl(alpha_path)
backdoor_records = load_jsonl(backdoor_path)


# ------------------------------------------------------------
# 10.14 Dataset parser coverage validation
# ------------------------------------------------------------

def validate_counterbench_parser(records, dataset_name):
    parsed_count = 0
    failed_questions = []
    type_counts = Counter()

    for record in records:
        question = record.get("question", "")

        parsed = parse_counterfactual_question(question)

        if parsed is not None:
            parsed_count += 1
            type_counts[parsed.query_type] += 1
        else:
            failed_questions.append({
                "question_id": record.get("question_id"),
                "question": question,
                "type": record.get("type")
            })

    total = len(records)
    coverage = parsed_count / total * 100 if total else 0

    print(f"\n{dataset_name}")
    print("-" * 50)
    print(f"Total questions: {total}")
    print(f"Successfully parsed: {parsed_count}")
    print(f"Failed questions: {len(failed_questions)}")
    print(f"Parser coverage: {coverage:.2f}%")
    print(f"Parsed question types: {dict(type_counts)}")

    return {
        "dataset": dataset_name,
        "total": total,
        "parsed": parsed_count,
        "failed": len(failed_questions),
        "coverage_percent": round(coverage, 2),
        "type_counts": dict(type_counts),
        "failed_questions": failed_questions
    }


alpha_stats = validate_counterbench_parser(
    alpha_records,
    "CounterBench Alpha"
)

backdoor_stats = validate_counterbench_parser(
    backdoor_records,
    "CounterBench Backdoor"
)


# ------------------------------------------------------------
# 10.15 Save validation results
# ------------------------------------------------------------

parser_validation = {
    "causal_mechanism_tests": "passed",
    "intervention_parser_tests": "passed",
    "question_parser_tests": "passed",
    "alpha": alpha_stats,
    "backdoor": backdoor_stats
}

validation_path = OUTPUT_DIR / "counterbench_parser_validation.json"

with open(validation_path, "w", encoding="utf-8") as file:
    json.dump(parser_validation, file, indent=2)

print("\n" + "=" * 60)
print("SECTION 10 COMPLETED")
print("=" * 60)
print(f"Validation report saved to: {validation_path}")


Causal mechanism parser tests: PASSED
Intervention parser tests: PASSED
Counterfactual question parser tests: PASSED

CounterBench Alpha
--------------------------------------------------
Total questions: 1000
Successfully parsed: 1000
Failed questions: 0
Parser coverage: 100.00%
Parsed question types: {'single': 500, 'joint': 250, 'nested': 250}

CounterBench Backdoor
--------------------------------------------------
Total questions: 200
Successfully parsed: 200
Failed questions: 0
Parser coverage: 100.00%
Parsed question types: {'single': 200}

SECTION 10 COMPLETED
Validation report saved to: /kaggle/working/AI_Assignment_2026/outputs/counterbench_parser_validation.json


In [10]:

# ============================================================
# SECTION 11: COUNTERBENCH CAUSAL REASONING ENGINE
# ============================================================

from dataclasses import dataclass, field
from collections import defaultdict
import re


# ------------------------------------------------------------
# 11.1 Causal graph representation
# ------------------------------------------------------------

@dataclass
class CausalGraph:
    mechanisms: list
    variables: set = field(default_factory=set)

    def __post_init__(self):
        self.variables = set(self.variables)

        for mechanism in self.mechanisms:
            self.variables.update(mechanism.parents)
            self.variables.add(mechanism.child)

    def add_mechanism(self, mechanism):
        self.mechanisms.append(mechanism)
        self.variables.update(mechanism.parents)
        self.variables.add(mechanism.child)

    def summary(self):
        return {
            "variables": len(self.variables),
            "mechanisms": len(self.mechanisms),
            "variables_list": sorted(self.variables)
        }


# ------------------------------------------------------------
# 11.2 Causal reasoning result
# ------------------------------------------------------------

@dataclass
class CausalReasoningResult:
    state: dict
    interventions: dict
    derived_assignments: dict
    conflicts: list
    activated_mechanisms: list
    iterations: int

    def outcome(self, variable):
        variable = normalize_variable(variable)
        return self.state.get(variable, False)

    @property
    def consistent(self):
        return len(self.conflicts) == 0



def mechanism_is_active(mechanism, state):
    """
    Evaluate a causal mechanism using the required truth value
    of each parent.

    AND: every parent must match its required value.
    OR: at least one parent must match its required value.
    """

    parent_matches = [
        state.get(parent, False) == required_value
        for parent, required_value in zip(
            mechanism.parents,
            mechanism.parent_values
        )
    ]

    if mechanism.parent_operator == "AND":
        return all(parent_matches)

    if mechanism.parent_operator == "OR":
        return any(parent_matches)

    raise ValueError(
        f"Unsupported operator: {mechanism.parent_operator}"
    )


def build_causal_graph(given_info):
    """
    Convert causal statements into a CausalGraph.

    Requires parse_causal_mechanisms() from Section 10,
    redefined below with corrected sentence separation.
    """

    mechanisms = parse_causal_mechanisms(given_info)

    return CausalGraph(mechanisms=mechanisms)


# ------------------------------------------------------------
# 11.5 Normalize intervention inputs
# ------------------------------------------------------------

def normalize_intervention_input(interventions):
    """
    Accept either:
        {"ziklo": False, "blaf": True}

    or a list/tuple of Intervention objects.
    """

    normalized = {}

    if interventions is None:
        return normalized

    if isinstance(interventions, dict):
        for variable, value in interventions.items():
            normalized[normalize_variable(variable)] = bool(value)

    else:
        for intervention in interventions:
            normalized[
                normalize_variable(intervention.variable)
            ] = bool(intervention.value)

    return normalized


# ------------------------------------------------------------
# 11.6 Forward causal propagation
# ------------------------------------------------------------

def simulate_causal_graph(
    graph,
    interventions=None,
    initial_state=None,
    max_iterations=100
):
    """
    Deterministic forward propagation.

    Rules:
      - All graph variables initially default to False.
      - Initial assignments can set starting values.
      - Interventions override initial assignments.
      - Activated mechanisms propose consequent values.
      - Explicit interventions take priority over causal effects.
      - Conflicting causal effects are recorded.
      - Each mechanism fires at most once.
    """

    if not isinstance(graph, CausalGraph):
        raise TypeError("graph must be a CausalGraph")

    intervention_map = normalize_intervention_input(interventions)

    # Initialize variables.
    state = {
        variable: False
        for variable in graph.variables
    }

    # Apply initial state.
    if initial_state:
        for variable, value in initial_state.items():
            state[normalize_variable(variable)] = bool(value)

    # Apply interventions after initial state.
    for variable, value in intervention_map.items():
        state[variable] = value

    derived_assignments = {}
    activated = set()
    conflicts = []
    conflict_keys = set()

    iterations = 0

    for iteration in range(max_iterations):
        iterations = iteration + 1
        new_effects = []

        # Find newly activated mechanisms.
        for mechanism_index, mechanism in enumerate(graph.mechanisms):

            if mechanism_index in activated:
                continue

            if mechanism_is_active(mechanism, state):
                activated.add(mechanism_index)

                new_effects.append(
                    (
                        mechanism_index,
                        mechanism.child,
                        mechanism.child_value
                    )
                )

        if not new_effects:
            break

        # Group proposed effects by consequent variable.
        effects_by_child = defaultdict(list)

        for mechanism_index, child, effect_value in new_effects:
            effects_by_child[child].append(
                (mechanism_index, effect_value)
            )

        # Resolve proposed effects.
        for child, effects in effects_by_child.items():

            # Interventions override causal effects.
            if child in intervention_map:
                continue

            proposed_values = {
                value for _, value in effects
            }

            # Simultaneous contradictory effects.
            if len(proposed_values) > 1:
                conflict_key = (
                    child,
                    tuple(sorted(proposed_values))
                )

                if conflict_key not in conflict_keys:
                    conflicts.append({
                        "variable": child,
                        "values": sorted(proposed_values),
                        "mechanism_indices": [
                            index for index, _ in effects
                        ]
                    })
                    conflict_keys.add(conflict_key)

                continue

            proposed_value = next(iter(proposed_values))

            # Check for conflict with an earlier assignment.
            if (
                child in derived_assignments
                and derived_assignments[child] != proposed_value
            ):
                conflict_key = (
                    child,
                    tuple(sorted({
                        derived_assignments[child],
                        proposed_value
                    }))
                )

                if conflict_key not in conflict_keys:
                    conflicts.append({
                        "variable": child,
                        "values": sorted({
                            derived_assignments[child],
                            proposed_value
                        }),
                        "mechanism_indices": [
                            index for index, _ in effects
                        ]
                    })
                    conflict_keys.add(conflict_key)

                continue

            derived_assignments[child] = proposed_value
            state[child] = proposed_value

    else:
        raise RuntimeError(
            "Causal propagation exceeded the maximum iteration limit."
        )

    return CausalReasoningResult(
        state=state,
        interventions=intervention_map,
        derived_assignments=derived_assignments,
        conflicts=conflicts,
        activated_mechanisms=sorted(activated),
        iterations=iterations
    )


# ------------------------------------------------------------
# 11.7 Counterfactual prediction
# ------------------------------------------------------------

def predict_counterfactual(given_info, question):
    """
    Predict the queried outcome under the specified interventions.
    """

    mechanisms = parse_causal_mechanisms(given_info)
    graph = CausalGraph(mechanisms=mechanisms)

    parsed_question = parse_counterfactual_question(question)

    if parsed_question is None:
        raise ValueError(
            f"Could not parse counterfactual question: {question}"
        )

    result = simulate_causal_graph(
        graph=graph,
        interventions=parsed_question.interventions
    )

    outcome = parsed_question.outcome

    prediction = (
        result.outcome(outcome)
        if result.consistent
        else None
    )

    return {
        "prediction": prediction,
        "outcome": outcome,
        "query_type": parsed_question.query_type,
        "interventions": {
            item.variable: item.value
            for item in parsed_question.interventions
        },
        "state": result.state,
        "derived_assignments": result.derived_assignments,
        "conflicts": result.conflicts,
        "activated_mechanisms": result.activated_mechanisms,
        "iterations": result.iterations,
        "consistent": result.consistent
    }


# ------------------------------------------------------------
# 11.8 Corrected causal statement parser
# ------------------------------------------------------------

def parse_causal_mechanisms(given_info):
    """
    Parse causal statements separated by:
      - Full stops
      - Newlines
      - Semicolons

    Supported:
      A causes B
      A causes not B
      A and B together cause C
      A and B cause C
      A or B causes C
      A causes B and C
      A causes B, C and D
    """

    mechanisms = []

    if not isinstance(given_info, str):
        return mechanisms

    # IMPORTANT FIX:
    # Split individual sentences before applying causal patterns.
    statements = re.split(
        r"(?<=[.!?])\s+|[\n;]+",
        given_info.strip()
    )

    for statement in statements:
        statement = statement.strip().strip(".;!?")

        if not statement:
            continue

        # Multiple antecedents: AND / OR.
        match = COMBINATION_CAUSE_PATTERN.match(statement)

        if match:
            parent1, operator, parent2, negation, child = match.groups()

            mechanisms.append(
                CausalMechanism(
                    parents=(
                        normalize_variable(parent1),
                        normalize_variable(parent2)
                    ),
                    parent_operator=operator.upper(),
                    child=normalize_variable(child),
                    child_value=not bool(negation)
                )
            )
            continue

        # Single antecedent with multiple effects.
        match = MULTIPLE_EFFECT_PATTERN.match(statement)

        if match:
            parent, effect_text = match.groups()

            effects = parse_effect_list(effect_text)

            for child, child_value in effects:
                mechanisms.append(
                    CausalMechanism(
                        parents=(normalize_variable(parent),),
                        parent_operator="AND",
                        child=child,
                        child_value=child_value
                    )
                )

    return mechanisms


# ------------------------------------------------------------
# 11.9 Test: single causal mechanism
# ------------------------------------------------------------

graph_single = build_causal_graph(
    "Ziklo causes Blaf"
)

result_single = simulate_causal_graph(
    graph_single,
    interventions={"ziklo": True}
)

assert result_single.outcome("ziklo") is True
assert result_single.outcome("blaf") is True
assert result_single.consistent

print("Single-cause propagation test: PASSED")


# ------------------------------------------------------------
# 11.10 Test: negative causal effect
# ------------------------------------------------------------

graph_negative = build_causal_graph(
    "Ziklo causes not Blaf"
)

result_negative = simulate_causal_graph(
    graph_negative,
    interventions={"ziklo": True}
)

assert result_negative.outcome("ziklo") is True
assert result_negative.outcome("blaf") is False
assert result_negative.consistent

print("Negative-effect propagation test: PASSED")


# ------------------------------------------------------------
# 11.11 Test: conjunctive cause
# ------------------------------------------------------------

graph_and = build_causal_graph(
    "Blaf and Trune together cause Vork"
)

result_and_incomplete = simulate_causal_graph(
    graph_and,
    interventions={"blaf": True}
)

result_and_complete = simulate_causal_graph(
    graph_and,
    interventions={
        "blaf": True,
        "trune": True
    }
)

assert result_and_incomplete.outcome("vork") is False
assert result_and_complete.outcome("vork") is True

print("Conjunctive-cause propagation test: PASSED")


# ------------------------------------------------------------
# 11.12 Test: disjunctive cause
# ------------------------------------------------------------

graph_or = build_causal_graph(
    "Trune or Vork causes Lumbo"
)

result_or_false = simulate_causal_graph(
    graph_or,
    interventions={}
)

result_or_true = simulate_causal_graph(
    graph_or,
    interventions={"trune": True}
)

assert result_or_false.outcome("lumbo") is False
assert result_or_true.outcome("lumbo") is True

print("Disjunctive-cause propagation test: PASSED")


# ------------------------------------------------------------
# 11.13 Test: multi-step causal chain
# ------------------------------------------------------------

chain_information = (
    "Ziklo causes Blaf. "
    "Blaf causes Trune. "
    "Trune causes Vork. "
    "Vork causes Lumbo."
)

graph_chain = build_causal_graph(chain_information)

result_chain = simulate_causal_graph(
    graph_chain,
    interventions={"ziklo": True}
)

assert result_chain.outcome("blaf") is True
assert result_chain.outcome("trune") is True
assert result_chain.outcome("vork") is True
assert result_chain.outcome("lumbo") is True
assert result_chain.consistent

print("Multi-step causal-chain test: PASSED")


# ------------------------------------------------------------
# 11.14 Test: intervention priority
# ------------------------------------------------------------

graph_intervention = build_causal_graph(
    "Ziklo causes Blaf. "
    "Blaf causes Trune."
)

result_intervention = simulate_causal_graph(
    graph_intervention,
    interventions={
        "ziklo": True,
        "blaf": False
    }
)

assert result_intervention.outcome("ziklo") is True
assert result_intervention.outcome("blaf") is False
assert result_intervention.outcome("trune") is False

print("Intervention-priority test: PASSED")


# ------------------------------------------------------------
# 11.15 Test: contradictory causal effects
# ------------------------------------------------------------

graph_conflict = build_causal_graph(
    "Ziklo causes Blaf. "
    "Trune causes not Blaf."
)

result_conflict = simulate_causal_graph(
    graph_conflict,
    interventions={
        "ziklo": True,
        "trune": True
    }
)

assert not result_conflict.consistent
assert len(result_conflict.conflicts) > 0

print("Contradictory-effect detection test: PASSED")


# ------------------------------------------------------------
# 11.16 Test: end-to-end counterfactual prediction
# ------------------------------------------------------------

test_question = (
    "Would Lumbo occur if not Ziklo instead of Ziklo?"
)

test_prediction = predict_counterfactual(
    chain_information,
    test_question
)

assert test_prediction["prediction"] is False
assert test_prediction["consistent"]

print("End-to-end counterfactual test: PASSED")


# ------------------------------------------------------------
# 11.17 Final Section 11 results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SECTION 11: CAUSAL REASONING ENGINE")
print("=" * 60)

print("\nCausal graph summary:")
print(graph_chain.summary())

print("\nEnd-to-end counterfactual prediction:")
print("Outcome:", test_prediction["outcome"])
print("Interventions:", test_prediction["interventions"])
print("Prediction:", test_prediction["prediction"])
print("Final state:", test_prediction["state"])
print("Activated mechanisms:",
      test_prediction["activated_mechanisms"])
print("Propagation iterations:", test_prediction["iterations"])

print("\nFinal causal chain state:")
print(result_chain.state)

print("\nSECTION 11 VALIDATION COMPLETED")


Single-cause propagation test: PASSED
Negative-effect propagation test: PASSED
Conjunctive-cause propagation test: PASSED
Disjunctive-cause propagation test: PASSED
Multi-step causal-chain test: PASSED
Intervention-priority test: PASSED
Contradictory-effect detection test: PASSED
End-to-end counterfactual test: PASSED

SECTION 11: CAUSAL REASONING ENGINE

Causal graph summary:
{'variables': 5, 'mechanisms': 4, 'variables_list': ['blaf', 'lumbo', 'trune', 'vork', 'ziklo']}

End-to-end counterfactual prediction:
Outcome: lumbo
Interventions: {'ziklo': False}
Prediction: False
Final state: {'blaf': False, 'lumbo': False, 'vork': False, 'ziklo': False, 'trune': False}
Activated mechanisms: []
Propagation iterations: 1

Final causal chain state:
{'blaf': True, 'lumbo': True, 'vork': True, 'ziklo': True, 'trune': True}

SECTION 11 VALIDATION COMPLETED


## Frozen CounterBench Alpha PBCI V1.0

In [11]:
# Read-only binding for the frozen Alpha dataset; no Alpha model/evaluator is run here.
alpha_df = COUNTERBENCH_ALPHA.copy()
print("CounterBench Alpha dataset loaded for reference:", len(alpha_df))
print("Frozen Alpha PBCI V1.0 evaluation is intentionally not rerun.")


CounterBench Alpha dataset loaded for reference: 1000
Frozen Alpha PBCI V1.0 evaluation is intentionally not rerun.


In [12]:

# ============================================================
# SECTION — COUNTERBENCH BACKDOOR V2.1
# Adapted faithfully from the supplied reasoning_agent notebook
# Includes actual inference-time measurement
# ============================================================

import re
import math
import time
import json
import itertools
from pathlib import Path
from collections import defaultdict

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Ensure paths are Path objects
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --------------------- Parsing utilities ---------------------

def _cb21_literal(token):
    """Convert a variable phrase to (variable, required_boolean_value)."""
    token = str(token).strip().lower().strip(" .;")

    negative = re.match(r"^not\s+(.+)$", token)

    if negative:
        variable = re.sub(r"^the\s+", "", negative.group(1)).strip()
        return variable, False

    variable = re.sub(r"^the\s+", "", token).strip()
    return variable, True


def cb21_parse(given_info):
    """
    Parse Boolean causal mechanisms and observations.

    A mechanism is stored by effect variable:
      effect -> [(AND/OR, [(parent, required_value), ...], effect_value)]

    Bernoulli text remains part of the context but is not used as a
    weighted prior. Root assignments are enumerated uniformly.
    """

    text = str(given_info).strip()

    mechanism_text, *observation_parts = re.split(
        r"\bwe observed\b",
        text,
        maxsplit=1,
        flags=re.I
    )

    mechanism_text = re.sub(
        r"^\s*we know that\s+",
        "",
        mechanism_text,
        flags=re.I
    )

    mechanisms = defaultdict(list)

    segments = [
        part.strip()
        for part in re.split(
            r",\s*(?:and\s+)?|\.\s*|;\s*",
            mechanism_text
        )
        if part.strip()
    ]

    for segment in segments:

        match = re.match(
            r"^(.+?)\s+(?:together\s+)?causes?\s+(not\s+)?(.+)$",
            segment,
            flags=re.I
        )

        if not match:
            continue

        cause_text, negative_effect, effect_text = match.groups()

        effect_text = re.sub(
            r"^the\s+",
            "",
            effect_text.strip().lower()
        ).strip(" .;")

        if re.search(r"\bor\b", cause_text, flags=re.I):
            operator = "or"
            parent_tokens = re.split(
                r"\s+or\s+",
                cause_text,
                flags=re.I
            )
        else:
            operator = "and"
            parent_tokens = re.split(
                r"\s+and\s+",
                cause_text,
                flags=re.I
            )

        parent_literals = [
            _cb21_literal(token)
            for token in parent_tokens
        ]

        if not effect_text or not parent_literals:
            continue

        mechanisms[effect_text].append(
            (
                operator,
                parent_literals,
                not bool(negative_effect)
            )
        )

    evidence = []

    if observation_parts:

        observation_text = observation_parts[0].strip().rstrip(".")

        for token in re.split(
            r"\s+and\s+|,\s*",
            observation_text,
            flags=re.I
        ):
            if token.strip():
                evidence.append(_cb21_literal(token))

    return dict(mechanisms), evidence


# --------------------- Causal evaluation ---------------------

def cb21_evaluate_variable(
    variable,
    mechanisms,
    root_assignment,
    interventions,
    memo,
    stack=()
):
    """Evaluate one variable recursively under a root assignment and do()."""

    if variable in interventions:
        return bool(interventions[variable])

    if variable in memo:
        return memo[variable]

    if variable not in mechanisms:
        return bool(root_assignment.get(variable, False))

    if variable in stack:
        return False

    rule_outputs = []

    for operator, parent_literals, effect_positive in mechanisms[variable]:

        parent_matches = [
            cb21_evaluate_variable(
                parent,
                mechanisms,
                root_assignment,
                interventions,
                memo,
                stack + (variable,)
            ) == required_value
            for parent, required_value in parent_literals
        ]

        active = (
            all(parent_matches)
            if operator == "and"
            else any(parent_matches)
        )

        rule_outputs.append(
            bool(active) == bool(effect_positive)
        )

    value = (
        any(rule_outputs)
        if len(rule_outputs) > 1
        else rule_outputs[0]
    )

    memo[variable] = bool(value)

    return bool(value)


# --------------------- Question patterns ---------------------

CB21_Q1 = re.compile(
    r"^\s*would\s+(.+?)\s+(?:still\s+)?occur\s+if\s+(.+?)"
    r"\s+instead of\s+(.+?)\s*\??\s*$",
    re.I
)

CB21_Q2 = re.compile(
    r"^\s*if\s+(.+?)\s+instead of\s+(.+?),?\s+would\s+(.+?)"
    r"\s+(?:still\s+)?occur\s*\??\s*$",
    re.I
)


# --------------------- Main reasoning function ---------------------

def cb21_answer(given_info, question, trace=False):
    """Pearl-style abduction, action, and prediction by root-world enumeration."""

    mechanisms, evidence = cb21_parse(given_info)

    match = CB21_Q1.match(str(question).strip())

    if match:
        target_text, intervention_text, factual_text = match.groups()

    else:
        match = CB21_Q2.match(str(question).strip())

        if not match:
            return None

        intervention_text, factual_text, target_text = match.groups()

    target_literals = [
        _cb21_literal(token)
        for token in re.split(
            r"\s+and\s+",
            target_text,
            flags=re.I
        )
    ]

    intervention_map = dict(
        _cb21_literal(token)
        for token in re.split(
            r"\s+and\s+",
            intervention_text,
            flags=re.I
        )
    )

    factual_literals = [
        _cb21_literal(token)
        for token in re.split(
            r"\s+and\s+",
            factual_text,
            flags=re.I
        )
    ]

    evidence_all = evidence + factual_literals

    effect_variables = set(mechanisms)

    parent_variables = {
        parent
        for rules in mechanisms.values()
        for _, literals, _ in rules
        for parent, _ in literals
    }

    roots = sorted(
        (
            parent_variables
            | {v for v, _ in evidence_all}
            | set(intervention_map)
        ) - effect_variables
    )

    if len(roots) > 14:
        raise ValueError(
            f"Root enumeration limit exceeded: {len(roots)} roots (max 14)."
        )

    compatible_worlds = 0
    positive_counterfactual_worlds = 0
    trace_rows = []

    for bits in itertools.product([False, True], repeat=len(roots)):

        assignment = dict(zip(roots, bits))

        factual_memo = {}

        evidence_holds = all(
            cb21_evaluate_variable(
                variable,
                mechanisms,
                assignment,
                {},
                factual_memo
            ) == required_value
            for variable, required_value in evidence_all
        )

        if not evidence_holds:
            continue

        compatible_worlds += 1

        counterfactual_memo = {}

        target_holds = all(
            cb21_evaluate_variable(
                variable,
                mechanisms,
                assignment,
                intervention_map,
                counterfactual_memo
            ) == required_value
            for variable, required_value in target_literals
        )

        if target_holds:
            positive_counterfactual_worlds += 1

        if trace:
            trace_rows.append({
                "root_assignment": assignment,
                "target_holds": target_holds
            })

    if compatible_worlds == 0:
        answer = False

    else:
        # Strict majority; a tie is predicted False.
        answer = (
            positive_counterfactual_worlds * 2 > compatible_worlds
        )

    if trace:
        return {
            "prediction": bool(answer),
            "compatible_worlds": compatible_worlds,
            "positive_worlds": positive_counterfactual_worlds,
            "roots": roots,
            "trace": trace_rows
        }

    return bool(answer)


# --------------------- Confidence interval ---------------------

def cb21_wilson_interval(successes, total, z=1.959963984540054):

    if total == 0:
        return (float("nan"), float("nan"))

    p = successes / total

    denom = 1 + z*z/total

    centre = (
        p + z*z/(2*total)
    ) / denom

    margin = (
        z * math.sqrt(
            (p*(1-p) + z*z/(4*total))/total
        ) / denom
    )

    return (
        max(0.0, centre-margin),
        min(1.0, centre+margin)
    )


# --------------------- Focused engine tests ---------------------

_test_context = (
    "We know that Blaf causes Ziklo, "
    "Ziklo or Blaf causes Trune, "
    "Trune causes Vork, and Vork causes Lumbo. "
    "Blaf~ Bern(0.2). We observed Trune"
)

_test_question = (
    "Would Lumbo occur if not Ziklo instead of Ziklo?"
)

_test_result = cb21_answer(
    _test_context,
    _test_question,
    trace=True
)

assert _test_result["compatible_worlds"] > 0
assert isinstance(_test_result["prediction"], bool)

assert cb21_answer(
    "We know that A causes B. We observed A",
    "Would B occur if not A instead of A?"
) is False

print("CounterBench Backdoor V2.1 parser/SCM smoke tests passed.")
print("Example compatible factual worlds:", _test_result["compatible_worlds"])
print("Example positive counterfactual worlds:", _test_result["positive_worlds"])


# ============================================================
# FULL COUNTERBENCH BACKDOOR V2.1 EVALUATION
# ============================================================

backdoor_df = COUNTERBENCH_BACKDOOR.copy()

backdoor_records = []
skipped_backdoor = []

# Actual reasoning/inference time only.
counterbench_inference_seconds = 0.0
counterbench_timed_examples = 0

for row_index, row in backdoor_df.iterrows():

    try:

        # Measure only the reasoning function execution.
        inference_start = time.perf_counter()

        prediction = cb21_answer(
            row["given_info"],
            row["question"]
        )

        inference_end = time.perf_counter()

        counterbench_inference_seconds += (
            inference_end - inference_start
        )

        counterbench_timed_examples += 1

        answer_text = str(row["answer"]).strip().lower()

        if (
            prediction is None
            or answer_text not in {"yes", "no", "true", "false"}
        ):

            skipped_backdoor.append({
                "row_index": int(row_index),
                "question_id": row.get("question_id"),
                "reason": "unparsed_question_or_label"
            })

            continue

        actual = answer_text in {"yes", "true"}

        backdoor_records.append({

            "row_index": int(row_index),
            "question_id": str(row.get("question_id")),
            "graph_id": row.get("graph_id"),
            "model_id": row.get("model_id"),
            "question": row["question"],
            "actual": bool(actual),
            "predicted": bool(prediction),
            "correct": bool(actual == prediction),
            "type": row.get("type", "backdoor")

        })

    except Exception as exc:

        skipped_backdoor.append({
            "row_index": int(row_index),
            "question_id": row.get("question_id"),
            "reason": repr(exc)
        })


backdoor_v21_predictions = pd.DataFrame(backdoor_records)


# --------------------- Evaluation validation ---------------------

if len(backdoor_v21_predictions) != len(backdoor_df):

    print(
        "WARNING: evaluated",
        len(backdoor_v21_predictions),
        "of",
        len(backdoor_df),
        "rows; skipped:",
        len(skipped_backdoor)
    )

else:

    print(
        "All Backdoor V2 rows evaluated:",
        len(backdoor_v21_predictions)
    )


if backdoor_v21_predictions.empty:
    raise RuntimeError(
        "CounterBench Backdoor V2.1 produced no predictions."
    )


y_true_v21 = (
    backdoor_v21_predictions["actual"]
    .astype(bool)
    .to_numpy()
)

y_pred_v21 = (
    backdoor_v21_predictions["predicted"]
    .astype(bool)
    .to_numpy()
)


# --------------------- Metrics ---------------------

cm_v21 = confusion_matrix(
    y_true_v21,
    y_pred_v21,
    labels=[False, True]
)

correct_v21 = int(
    (y_true_v21 == y_pred_v21).sum()
)

ci_v21 = cb21_wilson_interval(
    correct_v21,
    len(y_true_v21)
)

average_inference_time = (
    counterbench_inference_seconds / counterbench_timed_examples
    if counterbench_timed_examples > 0
    else float("nan")
)

metrics_v21 = {

    "dataset": "CounterBench Backdoor V2",

    "method": (
        "V2.1 reference SCM root-world abduction "
        "+ intervention + prediction"
    ),

    "examples": int(len(y_true_v21)),

    "correct": correct_v21,

    "accuracy": float(
        accuracy_score(y_true_v21, y_pred_v21)
    ),

    "accuracy_wilson_95_ci": [
        float(ci_v21[0]),
        float(ci_v21[1])
    ],

    "precision": float(
        precision_score(
            y_true_v21,
            y_pred_v21,
            zero_division=0
        )
    ),

    "recall": float(
        recall_score(
            y_true_v21,
            y_pred_v21,
            zero_division=0
        )
    ),

    "f1": float(
        f1_score(
            y_true_v21,
            y_pred_v21,
            zero_division=0
        )
    ),

    "confusion_matrix_tn_fp_fn_tp": cm_v21.tolist(),

    "skipped": int(len(skipped_backdoor)),

    # NEW: Runtime measurements
    "runtime_seconds": float(counterbench_inference_seconds),

    "average_inference_seconds_per_example": float(
        average_inference_time
    ),

    "timed_examples": int(counterbench_timed_examples)

}


# --------------------- Print results ---------------------

print("\n" + "=" * 78)
print("COUNTERBENCH BACKDOOR V2.1 — EVALUATION")
print("=" * 78)

print("Examples:", metrics_v21["examples"])
print("Correct:", metrics_v21["correct"])

print(f"Accuracy:  {metrics_v21['accuracy']:.4f}")
print(f"Precision: {metrics_v21['precision']:.4f}")
print(f"Recall:    {metrics_v21['recall']:.4f}")
print(f"F1-score:  {metrics_v21['f1']:.4f}")

print(
    "95% Wilson CI:",
    tuple(round(x, 4) for x in ci_v21)
)

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(cm_v21)

print("\nINFERENCE TIME")
print("-" * 45)

print(
    f"Total inference time: "
    f"{counterbench_inference_seconds:.6f} seconds"
)

print(
    f"Timed examples: {counterbench_timed_examples}"
)

print(
    f"Average time per example: "
    f"{average_inference_time:.6f} seconds"
)


# --------------------- Frozen baseline comparison ---------------------

baseline_path = OUTPUT_DIR / "counterbench_backdoor_predictions.csv"

if baseline_path.exists():

    baseline_df = pd.read_csv(baseline_path)

    required_baseline_cols = {
        "question_id",
        "actual",
        "predicted"
    }

    if required_baseline_cols.issubset(baseline_df.columns):

        baseline_df["question_id"] = (
            baseline_df["question_id"].astype(str)
        )

        comparison = backdoor_v21_predictions.merge(

            baseline_df[
                ["question_id", "actual", "predicted"]
            ].rename(
                columns={
                    "actual": "baseline_actual",
                    "predicted": "baseline_predicted"
                }
            ),

            on="question_id",
            how="inner",
            validate="one_to_one"
        )

        if len(comparison) == len(backdoor_v21_predictions):

            baseline_correct = int(
                (
                    comparison["actual"].astype(bool)
                    == comparison["baseline_predicted"].astype(bool)
                ).sum()
            )

            baseline_ci = cb21_wilson_interval(
                baseline_correct,
                len(comparison)
            )

            print("\nFROZEN BASELINE vs V2.1 (matched question IDs)")

            print(
                "Baseline accuracy:",
                f"{baseline_correct/len(comparison):.4f}"
            )

            print(
                "Baseline 95% Wilson CI:",
                tuple(round(x, 4) for x in baseline_ci)
            )

            print(
                "V2.1 accuracy:",
                f"{metrics_v21['accuracy']:.4f}"
            )

            print(
                "Absolute change (percentage points):",
                round(
                    100 * (
                        metrics_v21["accuracy"]
                        - baseline_correct / len(comparison)
                    ),
                    2
                )
            )

            metrics_v21["matched_baseline"] = {

                "examples": int(len(comparison)),

                "accuracy": float(
                    baseline_correct / len(comparison)
                ),

                "accuracy_wilson_95_ci": [
                    float(baseline_ci[0]),
                    float(baseline_ci[1])
                ],

                "accuracy_change_percentage_points": float(
                    100 * (
                        metrics_v21["accuracy"]
                        - baseline_correct / len(comparison)
                    )
                )
            }

        else:

            print(
                "\nBaseline comparison skipped: "
                "question IDs did not align 1:1."
            )

    else:

        print(
            "\nBaseline comparison skipped: "
            "expected columns not found."
        )

else:

    print(
        "\nFrozen baseline file not found; "
        "V2.1 metrics still saved separately."
    )


# --------------------- Save outputs ---------------------

# Separate filenames preserve the frozen baseline.

v21_prediction_path = (
    OUTPUT_DIR / "counterbench_backdoor_v21_predictions.csv"
)

v21_metrics_path = (
    OUTPUT_DIR / "counterbench_backdoor_v21_metrics.json"
)

v21_errors_path = (
    OUTPUT_DIR / "counterbench_backdoor_v21_errors.csv"
)


backdoor_v21_predictions.to_csv(
    v21_prediction_path,
    index=False
)

backdoor_v21_predictions.loc[
    ~backdoor_v21_predictions["correct"]
].to_csv(
    v21_errors_path,
    index=False
)


with open(
    v21_metrics_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics_v21,
        f,
        indent=2
    )


if skipped_backdoor:

    pd.DataFrame(skipped_backdoor).to_csv(
        OUTPUT_DIR / "counterbench_backdoor_v21_skipped.csv",
        index=False
    )


print("\nV2.1 outputs saved:")

print(" -", v21_prediction_path)
print(" -", v21_metrics_path)
print(" -", v21_errors_path)

print("\nRuntime successfully recorded in metrics JSON.")

print(
    "\nFrozen CounterBench Alpha PBCI V1.0 and "
    "Backdoor V2 baseline files were not modified."
)


CounterBench Backdoor V2.1 parser/SCM smoke tests passed.
Example compatible factual worlds: 1
Example positive counterfactual worlds: 1
All Backdoor V2 rows evaluated: 200

COUNTERBENCH BACKDOOR V2.1 — EVALUATION
Examples: 200
Correct: 180
Accuracy:  0.9000
Precision: 0.9412
Recall:    0.8421
F1-score:  0.8889
95% Wilson CI: (0.8506, 0.9343)

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[100   5]
 [ 15  80]]

INFERENCE TIME
---------------------------------------------
Total inference time: 0.028274 seconds
Timed examples: 200
Average time per example: 0.000141 seconds

Frozen baseline file not found; V2.1 metrics still saved separately.

V2.1 outputs saved:
 - /kaggle/working/AI_Assignment_2026/outputs/counterbench_backdoor_v21_predictions.csv
 - /kaggle/working/AI_Assignment_2026/outputs/counterbench_backdoor_v21_metrics.json
 - /kaggle/working/AI_Assignment_2026/outputs/counterbench_backdoor_v21_errors.csv

Runtime successfully recorded in metrics JSON.

Frozen CounterBench Alpha PBCI

## FOLIO — Z3 first-order entailment

In [13]:

%pip install -q z3-solver

import z3

print("=" * 70)
print("Z3 INSTALLATION VERIFICATION")
print("=" * 70)

print("Z3 version:", z3.get_version_string())

# Basic solver test
x = z3.Int("x")

solver = z3.Solver()
solver.add(x > 5)
solver.add(x < 10)

print("Solver test:", solver.check())
print("Example model:", solver.model())

print("\nZ3 is ready for the FOLIO reasoning component.")


Note: you may need to restart the kernel to use updated packages.
Z3 INSTALLATION VERIFICATION
Z3 version: 5.4.1
Solver test: sat
Example model: [x = 6]

Z3 is ready for the FOLIO reasoning component.


In [14]:
# ============================================================
# CELL 17: FOLIO — Z3 FIRST-ORDER LOGIC REASONING
# FINAL FROZEN IMPLEMENTATION
# ============================================================
#
# Method:
# 1. Parse FOL formulas into an AST.
# 2. Apply conservative fuzzy alignment.
# 3. Translate formulas into Z3 first-order logic.
# 4. Check premise consistency.
# 5. Check entailment using Premises AND NOT Conclusion.
# 6. Map labels: True -> True; False/Uncertain -> False.
#
# Final parser recovery:
# - Only repairs missing closing parentheses at end-of-input.
# - Does not modify logical operators, predicates, arguments,
#   or quantifier scopes.
# - Unrecoverable formulas remain parser errors.
#
# Final evaluation:
# - Reuses existing predictions if available.
# - Reports parser coverage and parser-clean metrics.
# ============================================================

import json
import os
import re
import time
from difflib import SequenceMatcher
from collections import Counter

import pandas as pd
import z3


# ============================================================
# 1. FOL AST NODE
# ============================================================

class FOLNode:
    def __init__(self, kind, val=None, children=None, var=None):
        self.kind = kind
        self.val = val
        self.children = children or []
        self.var = var


# ============================================================
# 2. FOLIO PROVER
# ============================================================

class FOLIOProver:
    """First-order entailment checker using Z3."""

    DEFAULT_TIMEOUT_MS = 8000
    DEFAULT_VARS = {"x", "y", "z", "w", "u", "v"}

    def __init__(self, timeout_ms=None, fuzzy_alignment=True):
        self.timeout_ms = (
            self.DEFAULT_TIMEOUT_MS
            if timeout_ms is None
            else int(timeout_ms)
        )

        self.fuzzy_alignment = bool(fuzzy_alignment)

        self.last_status = "not_run"
        self.last_reason = ""
        self.last_solver_reason = ""

    # --------------------------------------------------------
    # 2.1 Clean FOL formula
    # --------------------------------------------------------

    @staticmethod
    def clean_fol_line(line):

        line = str(line).strip()

        line = re.sub(
            r"(?:Never:|Simeng:|\[).*$",
            "",
            line
        ).strip()

        line = line.replace("—>", "→")
        line = line.replace("->", "→")
        line = line.replace("⟷", "↔")

        line = re.sub(
            r"\b([A-Za-z0-9_.-]+)\s*=\s*([A-Za-z0-9_.-]+)\b",
            r"Equal(\1, \2)",
            line
        )

        return line

    # --------------------------------------------------------
    # 2.2 Tokenizer
    # --------------------------------------------------------

    @classmethod
    def tokenize(cls, text):

        text = cls.clean_fol_line(text)

        token_spec = re.compile(
            r"\s*(?:(∀|∃)\s*([A-Za-z_][A-Za-z0-9_]*)"
            r"|(→|↔|⊕|∧|∨|¬|\(|\)|,)"
            r"|([A-Za-z0-9_.\u00C0-\u024F-]+))"
        )

        tokens = []
        pos = 0

        while pos < len(text):

            match = token_spec.match(text, pos)

            if not match:

                if text[pos:].strip() == "":
                    break

                raise ValueError(
                    f"Unrecognized FOL syntax near: "
                    f"{text[pos:pos+30]!r}"
                )

            quantifier, qvar, symbol, identifier = match.groups()

            if quantifier:

                tokens.append(
                    ("QUANT", (quantifier, qvar.lower()))
                )

            elif symbol:

                tokens.append((symbol, symbol))

            else:

                tokens.append(("ID", identifier))

            pos = match.end()

        return tokens

    # --------------------------------------------------------
    # 2.3 Recursive descent parser
    # --------------------------------------------------------

    @classmethod
    def parse_formula(cls, text):

        tokens = cls.tokenize(text)

        if not tokens:
            raise ValueError("Empty FOL formula")

        pos = 0

        def peek():

            return (
                tokens[pos]
                if pos < len(tokens)
                else (None, None)
            )

        def consume(expected=None):

            nonlocal pos

            token = peek()

            if expected is not None and token[0] != expected:

                raise ValueError(
                    f"Expected {expected}, found {token[0]} "
                    f"in {text!r}"
                )

            pos += 1

            return token

        def parse_iff():

            node = parse_imp()

            while peek()[0] == "↔":

                consume()

                node = FOLNode(
                    "IFF",
                    children=[node, parse_imp()]
                )

            return node

        def parse_imp():

            left = parse_xor()

            if peek()[0] == "→":

                consume()

                return FOLNode(
                    "IMP",
                    children=[left, parse_imp()]
                )

            return left

        def parse_xor():

            node = parse_or()

            while peek()[0] == "⊕":

                consume()

                node = FOLNode(
                    "XOR",
                    children=[node, parse_or()]
                )

            return node

        def parse_or():

            node = parse_and()

            while peek()[0] == "∨":

                consume()

                node = FOLNode(
                    "OR",
                    children=[node, parse_and()]
                )

            return node

        def parse_and():

            node = parse_unary()

            while peek()[0] == "∧":

                consume()

                node = FOLNode(
                    "AND",
                    children=[node, parse_unary()]
                )

            return node

        def parse_unary():

            token_type, token_value = peek()

            if token_type == "¬":

                consume()

                return FOLNode(
                    "NOT",
                    children=[parse_unary()]
                )

            if token_type == "QUANT":

                consume()

                quantifier, variable = token_value

                return FOLNode(
                    "ALL" if quantifier == "∀" else "EXISTS",
                    children=[parse_unary()],
                    var=variable
                )

            if token_type == "(":

                consume("(")

                node = parse_iff()

                consume(")")

                return node

            if token_type == "ID":

                identifier = consume("ID")[1]

                lowered = identifier.lower()

                if lowered == "true":

                    return FOLNode("BOOL", val=True)

                if lowered == "false":

                    return FOLNode("BOOL", val=False)

                args = []

                if peek()[0] == "(":

                    consume("(")

                    if peek()[0] != ")":

                        while True:

                            arg_type, arg_value = peek()

                            if arg_type != "ID":

                                raise ValueError(
                                    "Expected term in argument list, "
                                    f"found {arg_type}"
                                )

                            args.append(
                                consume("ID")[1].lower()
                            )

                            if peek()[0] == ",":

                                consume(",")

                                continue

                            break

                    consume(")")

                return FOLNode(
                    "ATOM",
                    val=(lowered, tuple(args))
                )

            raise ValueError(
                f"Unexpected token {token_type} in {text!r}"
            )

        result = parse_iff()

        if pos != len(tokens):

            raise ValueError(
                f"Unparsed trailing tokens in {text!r}: "
                f"{tokens[pos:]}"
            )

        return result

    # --------------------------------------------------------
    # 2.4 Collect predicates and constants
    # --------------------------------------------------------

    @classmethod
    def collect_signature(cls, nodes):

        predicates = set()
        constants = set()

        def visit(node, bound):

            if node.kind in ("ALL", "EXISTS"):

                visit(
                    node.children[0],
                    bound | {node.var}
                )

            elif node.kind == "ATOM":

                pred, args = node.val

                if pred == "equal" and len(args) == 2:

                    for arg in args:

                        if (
                            arg not in bound
                            and arg not in cls.DEFAULT_VARS
                        ):

                            constants.add(arg)

                else:

                    predicates.add((pred, len(args)))

                    for arg in args:

                        if (
                            arg not in bound
                            and arg not in cls.DEFAULT_VARS
                        ):

                            constants.add(arg)

            else:

                for child in node.children:

                    visit(child, bound)

        for node in nodes:

            visit(node, set())

        return predicates, constants

    # --------------------------------------------------------
    # 2.5 Conservative fuzzy alignment
    # --------------------------------------------------------

    @classmethod
    def align_node_with_kb(
        cls,
        node,
        kb_predicates,
        kb_constants
    ):

        if node.kind == "ATOM":

            pred, args = node.val

            aligned_args = []

            for arg in args:

                if (
                    arg in kb_constants
                    or arg in cls.DEFAULT_VARS
                ):

                    aligned_args.append(arg)

                    continue

                best_constant = arg
                best_score = 0.0

                for candidate in sorted(kb_constants):

                    score = SequenceMatcher(
                        None,
                        arg,
                        candidate
                    ).ratio()

                    if score > best_score:

                        best_constant = candidate
                        best_score = score

                aligned_args.append(
                    best_constant
                    if best_score >= 0.72
                    else arg
                )

            if (
                pred != "equal"
                and (pred, len(aligned_args)) not in kb_predicates
            ):

                best_predicate = pred
                best_score = 0.0

                for candidate, arity in sorted(kb_predicates):

                    if arity != len(aligned_args):
                        continue

                    score = SequenceMatcher(
                        None,
                        pred,
                        candidate
                    ).ratio()

                    if score > best_score:

                        best_predicate = candidate
                        best_score = score

                if best_score >= 0.68:

                    pred = best_predicate

            return FOLNode(
                "ATOM",
                val=(pred, tuple(aligned_args))
            )

        if node.kind == "BOOL":

            return node

        return FOLNode(
            node.kind,
            node.val,
            [
                cls.align_node_with_kb(
                    child,
                    kb_predicates,
                    kb_constants
                )
                for child in node.children
            ],
            node.var
        )

    # --------------------------------------------------------
    # 2.6 Translate AST into Z3
    # --------------------------------------------------------

    @staticmethod
    def _translate(
        node,
        sort,
        predicates,
        constants,
        env,
        fresh_counter
    ):

        kind = node.kind

        if kind == "BOOL":

            return z3.BoolVal(bool(node.val))

        if kind == "ATOM":

            predicate, args = node.val

            if predicate == "equal":

                if len(args) != 2:

                    raise ValueError(
                        "Equal must have exactly two arguments"
                    )

                left = FOLIOProver._term(
                    args[0], sort, constants, env
                )

                right = FOLIOProver._term(
                    args[1], sort, constants, env
                )

                return left == right

            terms = [
                FOLIOProver._term(
                    arg, sort, constants, env
                )
                for arg in args
            ]

            key = (predicate, len(terms))

            if key not in predicates:

                if len(terms) == 0:

                    predicates[key] = z3.Bool(
                        f"pred_{predicate}_0"
                    )

                else:

                    predicates[key] = z3.Function(
                        f"pred_{predicate}_{len(terms)}",
                        *([sort] * len(terms)),
                        z3.BoolSort()
                    )

            if not terms:

                return predicates[key]

            return predicates[key](*terms)

        if kind == "NOT":

            return z3.Not(
                FOLIOProver._translate(
                    node.children[0],
                    sort,
                    predicates,
                    constants,
                    env,
                    fresh_counter
                )
            )

        if kind in ("AND", "OR", "IMP", "IFF", "XOR"):

            left = FOLIOProver._translate(
                node.children[0],
                sort,
                predicates,
                constants,
                env,
                fresh_counter
            )

            right = FOLIOProver._translate(
                node.children[1],
                sort,
                predicates,
                constants,
                env,
                fresh_counter
            )

            if kind == "AND":

                return z3.And(left, right)

            if kind == "OR":

                return z3.Or(left, right)

            if kind == "IMP":

                return z3.Implies(left, right)

            if kind == "IFF":

                return left == right

            return z3.Xor(left, right)

        if kind in ("ALL", "EXISTS"):

            fresh_counter[0] += 1

            variable = z3.Const(
                f"bound_{node.var}_{fresh_counter[0]}",
                sort
            )

            new_env = dict(env)

            new_env[node.var] = variable

            body = FOLIOProver._translate(
                node.children[0],
                sort,
                predicates,
                constants,
                new_env,
                fresh_counter
            )

            if kind == "ALL":

                return z3.ForAll([variable], body)

            return z3.Exists([variable], body)

        raise ValueError(
            f"Unsupported AST node: {kind}"
        )

    # --------------------------------------------------------
    # 2.7 Resolve constants and variables
    # --------------------------------------------------------

    @staticmethod
    def _term(name, sort, constants, env):

        if name in env:

            return env[name]

        if name not in constants:

            constants[name] = z3.Const(
                f"const_{name}",
                sort
            )

        return constants[name]

    # --------------------------------------------------------
    # 2.8 Main entailment procedure
    # --------------------------------------------------------

    def prove_folio_detailed(
        self,
        premises_fol_str,
        conclusion_fol_str
    ):

        self.last_status = "not_run"
        self.last_reason = ""
        self.last_solver_reason = ""

        if (
            not str(premises_fol_str).strip()
            or not str(conclusion_fol_str).strip()
        ):

            self.last_status = "parse_error"
            self.last_reason = (
                "Missing premises or conclusion"
            )

            return False, self.last_status

        try:

            premise_lines = [
                line.strip()
                for line in str(premises_fol_str).splitlines()
                if line.strip()
            ]

            premise_nodes = [
                self.parse_formula(line)
                for line in premise_lines
            ]

            conclusion_node = self.parse_formula(
                str(conclusion_fol_str).strip()
            )

            kb_predicates, kb_constants = (
                self.collect_signature(premise_nodes)
            )

            if self.fuzzy_alignment:

                conclusion_node = self.align_node_with_kb(
                    conclusion_node,
                    kb_predicates,
                    kb_constants
                )

            domain_sort = z3.DeclareSort(
                "FOLIO_Domain"
            )

            predicates = {}
            constants = {}
            fresh_counter = [0]

            translated_premises = [
                self._translate(
                    node,
                    domain_sort,
                    predicates,
                    constants,
                    {},
                    fresh_counter
                )
                for node in premise_nodes
            ]

            translated_conclusion = self._translate(
                conclusion_node,
                domain_sort,
                predicates,
                constants,
                {},
                fresh_counter
            )

            # First solver: check premise consistency.
            kb_solver = z3.Solver()

            kb_solver.set(timeout=self.timeout_ms)
            kb_solver.set(mbqi=True)

            kb_solver.add(*translated_premises)

            kb_result = kb_solver.check()

            if kb_result == z3.unsat:

                self.last_status = "inconsistent_kb"
                self.last_reason = (
                    "Premises are inconsistent"
                )

                return False, self.last_status

            if kb_result == z3.unknown:

                self.last_status = "unknown_kb"

                self.last_solver_reason = (
                    kb_solver.reason_unknown()
                )

                self.last_reason = (
                    "Z3 could not determine premise consistency"
                )

                return False, self.last_status

            # Second solver: test entailment.
            entailment_solver = z3.Solver()

            entailment_solver.set(timeout=self.timeout_ms)
            entailment_solver.set(mbqi=True)

            entailment_solver.add(
                *translated_premises
            )

            entailment_solver.add(
                z3.Not(translated_conclusion)
            )

            result = entailment_solver.check()

            if result == z3.unsat:

                self.last_status = "entailed"

                return True, self.last_status

            if result == z3.sat:

                self.last_status = "not_entailed"

                return False, self.last_status

            self.last_status = "unknown"

            self.last_solver_reason = (
                entailment_solver.reason_unknown()
            )

            self.last_reason = (
                "Z3 returned unknown; conservatively mapped to False"
            )

            return False, self.last_status

        except Exception as exc:

            self.last_status = "parse_or_translation_error"

            self.last_reason = repr(exc)

            return False, self.last_status

    def prove_folio(
        self,
        premises_fol_str,
        conclusion_fol_str
    ):

        prediction, _ = self.prove_folio_detailed(
            premises_fol_str,
            conclusion_fol_str
        )

        return prediction


# ============================================================
# 3. CONSERVATIVE PARSER RECOVERY
# ============================================================
#
# Only attempts to close unmatched parentheses at end-of-input.
#
# Does not modify:
# - Logical operators
# - Predicate names
# - Arguments
# - Variables
# - Quantifier scopes
# - Formula ordering
#
# Unrecoverable formulas remain parser errors.
# ============================================================

if "FOLIO_PARSE_RECOVERY_COUNTS" not in globals():

    FOLIO_PARSE_RECOVERY_COUNTS = {
        "repaired": 0,
        "unrepaired": 0
    }

if "FOLIO_PARSE_RECOVERY_LOG" not in globals():

    FOLIO_PARSE_RECOVERY_LOG = []

if "_FOLIO_PARSE_FORMULA_STRICT" not in globals():

    _FOLIO_PARSE_FORMULA_STRICT = (
        FOLIOProver.parse_formula.__func__
    )


def _folio_parse_formula_conservative(cls, text):

    try:

        return _FOLIO_PARSE_FORMULA_STRICT(
            cls,
            text
        )

    except ValueError as original_error:

        error_text = str(original_error)

        # Only recover missing closing parentheses at EOF.
        if "Expected ), found None" not in error_text:

            FOLIO_PARSE_RECOVERY_COUNTS[
                "unrepaired"
            ] += 1

            raise

        cleaned = cls.clean_fol_line(text)

        balance = 0
        minimum_balance = 0

        for character in cleaned:

            if character == "(":

                balance += 1

            elif character == ")":

                balance -= 1

                minimum_balance = min(
                    minimum_balance,
                    balance
                )

        # Extra closing parentheses are not modified.
        # Recovery is limited to 1–3 missing closing parentheses.
        if (
            minimum_balance < 0
            or balance < 1
            or balance > 3
        ):

            FOLIO_PARSE_RECOVERY_COUNTS[
                "unrepaired"
            ] += 1

            raise original_error

        repaired_text = cleaned + (")" * balance)

        try:

            parsed = _FOLIO_PARSE_FORMULA_STRICT(
                cls,
                repaired_text
            )

        except Exception:

            FOLIO_PARSE_RECOVERY_COUNTS[
                "unrepaired"
            ] += 1

            raise original_error

        FOLIO_PARSE_RECOVERY_COUNTS[
            "repaired"
        ] += 1

        FOLIO_PARSE_RECOVERY_LOG.append({

            "original": cleaned,

            "repaired": repaired_text,

            "added_closing_parentheses": balance

        })

        return parsed


FOLIOProver.parse_formula = classmethod(
    _folio_parse_formula_conservative
)

print(
    "Conservative FOLIO parser recovery enabled."
)


# ============================================================
# 4. LABEL MAPPING
# ============================================================

def folio_gold_binary(label):

    """True -> True; False/Uncertain -> False."""

    return (
        str(label).strip().lower() == "true"
    )


# ============================================================
# 5. FOLIO EVALUATION FUNCTION
# ============================================================

def evaluate_folio_split(
    dataframe,
    split_name,
    output_dir,
    timeout_ms=8000,
    fuzzy_alignment=True
):

    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

    prover = FOLIOProver(
        timeout_ms=timeout_ms,
        fuzzy_alignment=fuzzy_alignment
    )

    y_true = []
    y_pred = []

    records = []

    statuses = Counter()

    start = time.perf_counter()

    for index, row in dataframe.iterrows():

        pred, status = prover.prove_folio_detailed(
            row["premises-FOL"],
            row["conclusion-FOL"]
        )

        gold = folio_gold_binary(
            row["label"]
        )

        statuses[status] += 1

        y_true.append(gold)
        y_pred.append(bool(pred))

        records.append({

            "example_id": row.get(
                "example_id",
                index
            ),

            "story_id": row.get(
                "story_id",
                ""
            ),

            "original_label": row["label"],

            "gold_binary": gold,

            "predicted_label": bool(pred),

            "correct": bool(pred) == gold,

            "solver_status": status,

            "solver_reason": prover.last_solver_reason,

            "error_reason": prover.last_reason,

            "conclusion_FOL": row["conclusion-FOL"]

        })

    elapsed = time.perf_counter() - start

    metrics = {

        "split": split_name,

        "examples": len(y_true),

        "accuracy": (
            accuracy_score(y_true, y_pred)
            if y_true else 0.0
        ),

        "precision": (
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if y_true else 0.0
        ),

        "recall": (
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if y_true else 0.0
        ),

        "f1": (
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
            if y_true else 0.0
        ),

        "runtime_seconds": elapsed,

        "status_counts": dict(statuses),

        "confusion_matrix": (
            confusion_matrix(
                y_true,
                y_pred
            ).tolist()
            if y_true else []
        )

    }

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    prediction_path = os.path.join(
        output_dir,
        f"folio_{split_name.lower()}_predictions.csv"
    )

    predictions_df = pd.DataFrame(records)

    predictions_df.to_csv(
        prediction_path,
        index=False
    )

    print("\n" + "=" * 78)

    print(
        f"FOLIO {split_name.upper()} — "
        "Z3 FIRST-ORDER ENTAILMENT"
    )

    print("=" * 78)

    print(
        "Examples:",
        metrics["examples"]
    )

    print(
        f"Accuracy:  {metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: {metrics['precision']:.4f}"
    )

    print(
        f"Recall:    {metrics['recall']:.4f}"
    )

    print(
        f"F1-score:  {metrics['f1']:.4f}"
    )

    print(
        f"Runtime:   {elapsed:.2f} seconds"
    )

    print(
        "\nSolver statuses:",
        dict(statuses)
    )

    print(
        "\nConfusion matrix [ [TN, FP], [FN, TP] ]:"
    )

    print(
        metrics["confusion_matrix"]
    )

    print("\nClassification report:")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["False", "True"],
            zero_division=0
        )
    )

    print(
        "\nPredictions saved:",
        prediction_path
    )

    return metrics, predictions_df


# ============================================================
# 6. KAGGLE DATA AND OUTPUT PATHS
# ============================================================

DATA_DIR = (
    "/kaggle/working/AI_Assignment_2026/"
    "data/normalized"
)

OUTPUT_DIR = (
    "/kaggle/working/AI_Assignment_2026/"
    "outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 7. LOAD FOLIO DATA
# ============================================================

if "folio_train_df" in globals():

    _folio_train_df = folio_train_df

elif "FOLIO_TRAIN" in globals():

    _folio_train_df = FOLIO_TRAIN

else:

    _folio_train_path = os.path.join(
        DATA_DIR,
        "folio_train.jsonl"
    )

    _folio_train_df = pd.read_json(
        _folio_train_path,
        lines=True
    )


if "folio_validation_df" in globals():

    _folio_validation_df = folio_validation_df

elif "FOLIO_VALIDATION" in globals():

    _folio_validation_df = FOLIO_VALIDATION

else:

    _folio_validation_path = os.path.join(
        DATA_DIR,
        "folio_validation.jsonl"
    )

    _folio_validation_df = pd.read_json(
        _folio_validation_path,
        lines=True
    )


# ============================================================
# 8. VERIFY INPUT DATA
# ============================================================

_required_folio_columns = {

    "premises-FOL",

    "conclusion-FOL",

    "label"

}

for _split_name, _split_df in [

    ("train", _folio_train_df),

    ("validation", _folio_validation_df)

]:

    _missing = (
        _required_folio_columns
        - set(_split_df.columns)
    )

    if _missing:

        raise ValueError(
            f"FOLIO {_split_name} is missing required "
            f"columns: {sorted(_missing)}"
        )


# ============================================================
# 9. REUSE EXISTING FROZEN PREDICTIONS WHEN AVAILABLE
# ============================================================

_required_prediction_columns = {

    "gold_binary",

    "predicted_label",

    "solver_status"

}

_reuse_existing_predictions = (

    "train_predictions" in globals()

    and "validation_predictions" in globals()

    and isinstance(
        train_predictions,
        pd.DataFrame
    )

    and isinstance(
        validation_predictions,
        pd.DataFrame
    )

    and _required_prediction_columns.issubset(
        train_predictions.columns
    )

    and _required_prediction_columns.issubset(
        validation_predictions.columns
    )

    and len(train_predictions) == len(
        _folio_train_df
    )

    and len(validation_predictions) == len(
        _folio_validation_df
    )

)


if _reuse_existing_predictions:

    print(
        "\nReusing existing FOLIO predictions; "
        "no solver evaluation will be rerun."
    )

    train_predictions = (
        train_predictions.copy()
    )

    validation_predictions = (
        validation_predictions.copy()
    )

    print(
        "Training prediction rows:",
        len(train_predictions)
    )

    print(
        "Validation prediction rows:",
        len(validation_predictions)
    )


else:

    print(
        "\nNo compatible frozen predictions found; "
        "running FOLIO evaluation once."
    )

    train_metrics, train_predictions = (
        evaluate_folio_split(
            _folio_train_df,
            "train",
            OUTPUT_DIR
        )
    )

    validation_metrics, validation_predictions = (
        evaluate_folio_split(
            _folio_validation_df,
            "validation",
            OUTPUT_DIR
        )
    )

    combined_metrics = pd.DataFrame([

        train_metrics,

        validation_metrics

    ])

    metrics_path = os.path.join(
        OUTPUT_DIR,
        "folio_z3_metrics.csv"
    )

    combined_metrics.to_csv(
        metrics_path,
        index=False
    )

    print(
        "\nSaved metrics:",
        metrics_path
    )


# ============================================================
# 10. SAVE EXISTING METRICS IF AVAILABLE
# ============================================================

if (
    "train_metrics" in globals()
    and "validation_metrics" in globals()
):

    combined_metrics = pd.DataFrame([

        train_metrics,

        validation_metrics

    ])

    metrics_path = os.path.join(
        OUTPUT_DIR,
        "folio_z3_metrics.csv"
    )

    combined_metrics.to_csv(
        metrics_path,
        index=False
    )

    print(
        "\nSaved metrics:",
        metrics_path
    )


# ============================================================
# 11. PARSER RECOVERY SUMMARY
# ============================================================

print(
    "\nParser recovery counts:",
    FOLIO_PARSE_RECOVERY_COUNTS
)

print(
    "Successfully repaired formulas:",
    len(FOLIO_PARSE_RECOVERY_LOG)
)


# ============================================================
# 12. FINAL PARSER-CLEAN METRICS
# ============================================================
#
# Parser-clean metrics exclude examples with
# parse_or_translation_error status.
#
# Other solver statuses, including inconsistent_kb and
# unknown, remain included because they were processed by
# the reasoning system and have recorded predictions.
#
# Parser coverage is reported separately.
# ============================================================

def report_parser_clean_metrics(
    predictions_df,
    split_name
):

    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score
    )

    total = len(predictions_df)

    parse_error_mask = (

        predictions_df["solver_status"]
        == "parse_or_translation_error"

    )

    clean_df = predictions_df.loc[
        ~parse_error_mask
    ].copy()

    parser_errors = int(
        parse_error_mask.sum()
    )

    parsed_count = len(clean_df)

    if parsed_count:

        y_true = (
            clean_df["gold_binary"]
            .astype(bool)
        )

        y_pred = (
            clean_df["predicted_label"]
            .astype(bool)
        )

        clean_metrics = {

            "clean_accuracy": accuracy_score(
                y_true,
                y_pred
            ),

            "clean_precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "clean_recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "clean_f1": f1_score(
                y_true,
                y_pred,
                zero_division=0
            )

        }

    else:

        clean_metrics = {

            "clean_accuracy": 0.0,

            "clean_precision": 0.0,

            "clean_recall": 0.0,

            "clean_f1": 0.0

        }

    coverage = (
        parsed_count / total
        if total else 0.0
    )

    result = {

        "split": split_name,

        "total_examples": total,

        "parsed_examples": parsed_count,

        "parser_failures": parser_errors,

        "parser_coverage": coverage,

        **clean_metrics

    }

    print("\n" + "=" * 78)

    print(
        f"FOLIO {split_name.upper()} — "
        "PARSER-CLEAN CLASSIFICATION METRICS"
    )

    print("=" * 78)

    print(
        f"Total examples:       {total}"
    )

    print(
        f"Successfully parsed:  {parsed_count}"
    )

    print(
        f"Parser failures:      {parser_errors}"
    )

    print(
        f"Parser coverage:      {coverage:.2%}"
    )

    print(
        f"Clean accuracy:       "
        f"{clean_metrics['clean_accuracy']:.4f}"
    )

    print(
        f"Clean precision:      "
        f"{clean_metrics['clean_precision']:.4f}"
    )

    print(
        f"Clean recall:         "
        f"{clean_metrics['clean_recall']:.4f}"
    )

    print(
        f"Clean F1-score:       "
        f"{clean_metrics['clean_f1']:.4f}"
    )

    return result


# ============================================================
# 13. GENERATE FINAL SUMMARY
# ============================================================

parser_clean_summary = pd.DataFrame([

    report_parser_clean_metrics(
        train_predictions,
        "train"
    ),

    report_parser_clean_metrics(
        validation_predictions,
        "validation"
    )

])


# ============================================================
# 14. SAVE FINAL PARSER-CLEAN METRICS
# ============================================================

parser_clean_path = os.path.join(
    OUTPUT_DIR,
    "folio_parser_clean_metrics.csv"
)

parser_clean_summary.to_csv(
    parser_clean_path,
    index=False
)

print(
    "\nParser-clean metrics saved:",
    parser_clean_path
)


# ============================================================
# 15. FINAL STATUS
# ============================================================

print(
    "\nFinal parser recovery counts:",
    FOLIO_PARSE_RECOVERY_COUNTS
)

print(
    "Successfully repaired formulas:",
    len(FOLIO_PARSE_RECOVERY_LOG)
)

print(
    "Unrepaired parser failures:",
    FOLIO_PARSE_RECOVERY_COUNTS["unrepaired"]
)

print(
    "\nFOLIO evaluation completed."
)

print(
    "Existing parser failures are retained and reported."
)

print(
    "No experimental terminal-parenthesis repair was applied."
)

print(
    "\nFOLIO FINAL RESULTS ARE FROZEN."
)

Conservative FOLIO parser recovery enabled.

No compatible frozen predictions found; running FOLIO evaluation once.

FOLIO TRAIN — Z3 FIRST-ORDER ENTAILMENT
Examples: 1001
Accuracy:  0.8132
Precision: 0.9036
Recall:    0.5799
F1-score:  0.7064
Runtime:   10.03 seconds

Solver statuses: {'entailed': 249, 'not_entailed': 641, 'parse_or_translation_error': 105, 'inconsistent_kb': 6}

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[589, 24], [163, 225]]

Classification report:
              precision    recall  f1-score   support

       False       0.78      0.96      0.86       613
        True       0.90      0.58      0.71       388

    accuracy                           0.81      1001
   macro avg       0.84      0.77      0.78      1001
weighted avg       0.83      0.81      0.80      1001


Predictions saved: /kaggle/working/AI_Assignment_2026/outputs/folio_train_predictions.csv

FOLIO VALIDATION — Z3 FIRST-ORDER ENTAILMENT
Examples: 203
Accuracy:  0.8473
Precision: 0.9020
Recall:    0.

In [15]:

# ============================================================
# SECTION 19 — PUBLIC EVALUATION + PRIVATE SUBMISSION
# FINAL FROZEN VERSION
# ============================================================
#
# Responsibilities:
# 1. Collect existing public evaluation predictions.
# 2. Generate combined public prediction CSV.
# 3. Generate detailed prediction and metric reports.
# 4. Support future private-test inference.
# 5. Validate private predictions before writing submission.csv.
#
# IMPORTANT:
# - Does not retrain or rerun public benchmark evaluations.
# - Does not fabricate private-test predictions.
# - Public submission is only a fallback until private data arrives.
# - Existing benchmark results remain frozen.
# ============================================================

from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

from tqdm.auto import tqdm


# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

PROJECT_DIR = Path(PROJECT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUBMISSION_DIR = Path(
    "/kaggle/working/submission"
)

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUBMISSION_PATH = (
    SUBMISSION_DIR / "submission.csv"
)

# Leave None until the instructor releases the private test file.
# Supported formats: CSV, JSONL, JSON.

PRIVATE_TEST_PATH = None

# Example:
# PRIVATE_TEST_PATH = "/kaggle/input/competition/test.csv"


# ============================================================
# 2. PRIVATE TEST LOADER
# ============================================================

def load_private_test(path):

    path = Path(path)

    if not path.is_file():

        raise FileNotFoundError(
            f"Private test file not found: {path}"
        )

    suffix = path.suffix.lower()

    if suffix == ".csv":

        return pd.read_csv(path)

    if suffix in {".jsonl", ".jl"}:

        return pd.read_json(
            path,
            lines=True
        )

    if suffix == ".json":

        return pd.read_json(path)

    raise ValueError(
        "Supported formats are .csv, .jsonl, and .json."
    )


# ============================================================
# 3. SAFE BOOLEAN NORMALIZATION
# ============================================================

def _as_bool_prediction(value):

    if isinstance(value, dict):

        value = value.get("prediction")

    if value is None:

        raise ValueError(
            "Reasoning engine returned no prediction."
        )

    if isinstance(value, (bool, np.bool_)):

        return bool(value)

    if isinstance(value, (int, np.integer)):

        if value in (0, 1):

            return bool(value)

    if isinstance(value, (float, np.floating)):

        if value in (0.0, 1.0):

            return bool(value)

    if isinstance(value, str):

        normalized = value.strip().lower()

        if normalized in {
            "true", "1", "yes", "entailed"
        }:

            return True

        if normalized in {
            "false", "0", "no", "not_entailed",
            "not entailed", "uncertain"
        }:

            return False

    raise ValueError(
        f"Cannot safely interpret prediction: {value!r}"
    )


# ============================================================
# 4. PRIVATE RECORD PREDICTION DISPATCHER
# ============================================================

def predict_private_record(row):

    """Dispatch one record to its corresponding classical reasoner."""

    record = (
        row.to_dict()
        if isinstance(row, pd.Series)
        else dict(row)
    )

    task = str(
        record.get("task", "")
    ).strip().lower()

    # Infer task from input columns when task is not provided.
    if not task:

        if {
            "context",
            "statement"
        }.issubset(record):

            task = "ruletaker"

        elif {
            "given_info",
            "question"
        }.issubset(record):

            task = "counterbench"

        elif {
            "premises-FOL",
            "conclusion-FOL"
        }.issubset(record):

            task = "folio"

        else:

            raise ValueError(
                "Cannot infer task. Expected one of these field pairs: "
                "(context, statement), "
                "(given_info, question), "
                "(premises-FOL, conclusion-FOL)."
            )

    # --------------------------------------------------------
    # RuleTaker
    # --------------------------------------------------------

    if task in {
        "ruletaker",
        "rule_taker",
        "rule-taker"
    }:

        result = predict_ruletaker(
            record["context"],
            record["statement"],
            method="forward"
        )

        return _as_bool_prediction(result)

    # --------------------------------------------------------
    # CounterBench
    # --------------------------------------------------------

    if task in {
        "counterbench",
        "counterbench_backdoor",
        "backdoor"
    }:

        result = cb21_answer(
            record["given_info"],
            record["question"]
        )

        return _as_bool_prediction(result)

    # --------------------------------------------------------
    # FOLIO
    # --------------------------------------------------------

    if task in {
        "folio",
        "folio_z3"
    }:

        prover = FOLIOProver(
            timeout_ms=8000,
            fuzzy_alignment=True
        )

        predicted, solver_status = (
            prover.prove_folio_detailed(
                record["premises-FOL"],
                record["conclusion-FOL"]
            )
        )

        return bool(predicted)

    raise ValueError(
        f"Unsupported task value: {task!r}"
    )


# ============================================================
# 5. PRIVATE SUBMISSION GENERATOR
# ============================================================

def create_submission(
    test_df,
    output_path,
    source_name="private"
):

    """Run inference and create the two-column submission CSV."""

    if test_df.empty:

        raise ValueError(
            "Test dataset is empty."
        )

    # Identify ID column.
    id_candidates = [
        "id",
        "example_id",
        "question_id"
    ]

    id_col = next(
        (
            column
            for column in id_candidates
            if column in test_df.columns
        ),
        None
    )

    if id_col is None:

        raise ValueError(
            "Test file must contain an ID column named one of: "
            + ", ".join(id_candidates)
        )

    if (
        test_df[id_col].isna().any()
        or test_df[id_col].duplicated().any()
    ):

        raise ValueError(
            "Test file contains missing or duplicate IDs."
        )

    # Validate required input fields.
    for index, row in test_df.iterrows():

        record = row.to_dict()

        task = str(
            record.get("task", "")
        ).strip().lower()

        if not task:

            if {
                "context",
                "statement"
            }.issubset(record):

                task = "ruletaker"

            elif {
                "given_info",
                "question"
            }.issubset(record):

                task = "counterbench"

            elif {
                "premises-FOL",
                "conclusion-FOL"
            }.issubset(record):

                task = "folio"

        required_fields = {

            "ruletaker": {
                "context",
                "statement"
            },

            "rule_taker": {
                "context",
                "statement"
            },

            "rule-taker": {
                "context",
                "statement"
            },

            "counterbench": {
                "given_info",
                "question"
            },

            "counterbench_backdoor": {
                "given_info",
                "question"
            },

            "backdoor": {
                "given_info",
                "question"
            },

            "folio": {
                "premises-FOL",
                "conclusion-FOL"
            },

            "folio_z3": {
                "premises-FOL",
                "conclusion-FOL"
            }

        }

        if task not in required_fields:

            raise ValueError(
                f"Unsupported or unidentified task at row {index}: {task!r}"
            )

        missing = (
            required_fields[task]
            - set(record.keys())
        )

        if missing:

            raise ValueError(
                f"Missing fields at row {index}: {sorted(missing)}"
            )

    # Run inference.
    start = time.perf_counter()

    predictions = []
    errors = []

    for index, row in tqdm(
        test_df.iterrows(),
        total=len(test_df),
        desc=f"{source_name} inference"
    ):

        try:

            prediction = predict_private_record(row)

            predictions.append(
                _as_bool_prediction(prediction)
            )

        except Exception as exc:

            errors.append({

                "row_index": int(index),

                "id": row[id_col],

                "error": (
                    f"{type(exc).__name__}: {exc}"
                )

            })

            predictions.append(None)

    elapsed = (
        time.perf_counter() - start
    )

    # Do not write a partial submission.
    if errors:

        error_path = Path(output_path).with_name(
            f"{source_name}_submission_errors.csv"
        )

        pd.DataFrame(errors).to_csv(
            error_path,
            index=False
        )

        raise RuntimeError(
            f"{len(errors)} rows could not be predicted. "
            f"No submission was written. Inspect: {error_path}"
        )

    # Construct submission.
    submission = pd.DataFrame({

        "id": test_df[id_col].values,

        "predicted_label": [
            "True" if prediction else "False"
            for prediction in predictions
        ]

    })

    # Final validation.
    if (
        len(submission) != len(test_df)
        or submission["id"].duplicated().any()
        or submission.isna().any().any()
    ):

        raise AssertionError(
            "Submission validation failed."
        )

    if not set(
        submission["predicted_label"].unique()
    ).issubset({"True", "False"}):

        raise AssertionError(
            "Submission contains an invalid label."
        )

    # Save only after all validation passes.
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    submission.to_csv(
        output_path,
        index=False
    )

    print(
        f"\n{source_name.upper()} INFERENCE SUMMARY"
    )

    print(
        "Rows:",
        len(submission)
    )

    print(
        "Accuracy: not applicable "
        "(test labels are not supplied to inference)"
    )

    print(
        f"Total inference time: {elapsed:.6f} seconds"
    )

    print(
        f"Average per example: "
        f"{elapsed / len(submission):.8f} seconds"
    )

    print(
        "Saved:",
        output_path
    )

    display(
        submission.head()
    )

    return submission


# ============================================================
# 6. PUBLIC EVALUATION OUTPUT GENERATOR
# ============================================================

def build_public_evaluation_outputs():

    """
    Collect already-computed labelled public predictions.

    No benchmark evaluation is rerun here.
    """

    detail_frames = []
    metric_rows = []

    # --------------------------------------------------------
    # 6.1 RULETAKER
    # --------------------------------------------------------

    if "RULETAKER_PREDICTIONS" in globals():

        for split_name in [
            "test",
            "ood_test"
        ]:

            values = RULETAKER_PREDICTIONS.get(
                (split_name, "forward")
            )

            source_df = RULETAKER_DF.get(
                split_name
            )

            if (
                values is None
                or source_df is None
            ):

                continue

            ids = (
                source_df["id"].tolist()
                if "id" in source_df.columns
                else list(source_df.index)
            )

            actual = [
                bool(x)
                for x in values["actual"]
            ]

            predicted = [
                bool(x)
                for x in values["predicted"]
            ]

            detail_frames.append(
                pd.DataFrame({

                    "task": "ruletaker",

                    "id": [
                        f"ruletaker_{split_name}_{x}"
                        for x in ids
                    ],

                    "gold_label": [
                        "True" if x else "False"
                        for x in actual
                    ],

                    "predicted_label": [
                        "True" if x else "False"
                        for x in predicted
                    ],

                    "correct": [
                        a == p
                        for a, p in zip(actual, predicted)
                    ],

                    "split": split_name

                })
            )

        if "results_df" in globals():

            selected = results_df[
                results_df["dataset"].isin([
                    "test",
                    "ood_test"
                ])
                & (
                    results_df["method"] == "forward"
                )
            ]

            for _, row in selected.iterrows():

                metric_rows.append({

                    "task": "ruletaker",

                    "split": row["dataset"],

                    "method": row["method"],

                    "examples": int(row["examples"]),

                    "accuracy": float(row["accuracy"]),

                    "runtime_seconds": float(
                        row["total_elapsed_seconds"]
                    )

                })

    # --------------------------------------------------------
    # 6.2 COUNTERBENCH BACKDOOR V2.1
    # --------------------------------------------------------

    if (
        "backdoor_v21_predictions" in globals()
        and not backdoor_v21_predictions.empty
    ):

        cb = backdoor_v21_predictions

        detail_frames.append(
            pd.DataFrame({

                "task": "counterbench_backdoor_v21",

                "id": [
                    f"counterbench_backdoor_{x}"
                    for x in cb["question_id"].astype(str)
                ],

                "gold_label": [
                    "True" if x else "False"
                    for x in cb["actual"].astype(bool)
                ],

                "predicted_label": [
                    "True" if x else "False"
                    for x in cb["predicted"].astype(bool)
                ],

                "correct": cb["correct"].astype(bool).tolist(),

                "split": "public_labeled_evaluation"

            })
        )

        metric_rows.append({

            "task": "counterbench_backdoor_v21",

            "split": "public_labeled_evaluation",

            "method": (
                "root-world abduction + intervention + prediction"
            ),

            "examples": int(
                metrics_v21["examples"]
            ),

            "accuracy": float(
                metrics_v21["accuracy"]
            ),

            "runtime_seconds": float(
                metrics_v21.get(
                    "runtime_seconds",
                    float("nan")
                )
            )

        })

    # --------------------------------------------------------
    # 6.3 FOLIO
    # --------------------------------------------------------

    if (
        "validation_predictions" in globals()
        and not validation_predictions.empty
    ):

        folio = validation_predictions.copy()

        # Overall validation metrics use all 203 examples.
        overall_accuracy = (
            folio["correct"].astype(bool).mean()
        )

        parser_error_mask = (
            folio["solver_status"]
            == "parse_or_translation_error"
        )

        parser_failures = int(
            parser_error_mask.sum()
        )

        total_examples = len(folio)

        parsed_examples = (
            total_examples - parser_failures
        )

        parser_coverage = (
            parsed_examples / total_examples
            if total_examples else 0.0
        )

        clean_folio = folio.loc[
            ~parser_error_mask
        ].copy()

        clean_accuracy = (
            clean_folio["correct"].astype(bool).mean()
            if not clean_folio.empty
            else float("nan")
        )

        # Preserve detailed predictions.
        detail_frames.append(
            pd.DataFrame({

                "task": "folio_z3",

                "id": [
                    f"folio_validation_{x}"
                    for x in folio["example_id"].astype(str)
                ],

                "gold_label": [
                    "True" if x else "False"
                    for x in folio["gold_binary"].astype(bool)
                ],

                "predicted_label": [
                    "True" if x else "False"
                    for x in folio["predicted_label"].astype(bool)
                ],

                "correct": (
                    folio["correct"].astype(bool).tolist()
                ),

                "split": "validation"

            })
        )

        # Report both overall and parser-clean results.
        metric_rows.append({

            "task": "folio_z3",

            "split": "validation",

            "method": "Z3 first-order entailment",

            "examples": total_examples,

            "accuracy": overall_accuracy,

            "parser_coverage": parser_coverage,

            "parsed_examples": parsed_examples,

            "parser_failures": parser_failures,

            "parser_clean_accuracy": clean_accuracy,

            "runtime_seconds": (
                float(validation_metrics["runtime_seconds"])
                if "validation_metrics" in globals()
                else float("nan")
            )

        })

    # --------------------------------------------------------
    # 6.4 CHECK PUBLIC PREDICTIONS
    # --------------------------------------------------------

    if not detail_frames:

        raise RuntimeError(
            "No public evaluation predictions were found. "
            "Run the evaluation cells before this cell."
        )

    detailed = pd.concat(
        detail_frames,
        ignore_index=True
    )

    # Validate combined public predictions.
    if (
        detailed["id"].isna().any()
        or detailed["id"].duplicated().any()
    ):

        raise ValueError(
            "Public evaluation contains missing or duplicate IDs."
        )

    public_submission = detailed[
        ["id", "predicted_label"]
    ].copy()

    if public_submission.isna().any().any():

        raise ValueError(
            "Public submission contains missing values."
        )

    # --------------------------------------------------------
    # 6.5 SAVE PUBLIC OUTPUTS
    # --------------------------------------------------------

    public_submission.to_csv(
        SUBMISSION_PATH,
        index=False
    )

    detailed.to_csv(
        OUTPUT_DIR / "public_test_predictions_detailed.csv",
        index=False
    )

    metrics_df = pd.DataFrame(
        metric_rows
    )

    metrics_df.to_csv(
        OUTPUT_DIR / "public_evaluation_summary.csv",
        index=False
    )

    # --------------------------------------------------------
    # 6.6 DISPLAY SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 88)

    print(
        "PUBLIC DATA EVALUATION SUMMARY"
    )

    print("=" * 88)

    if not metrics_df.empty:

        display(metrics_df)

    print(
        "Public prediction rows:",
        len(public_submission)
    )

    print(
        "Public predictions saved to:",
        SUBMISSION_PATH
    )

    print(
        "Detailed labelled predictions:",
        OUTPUT_DIR / "public_test_predictions_detailed.csv"
    )

    print(
        "Metrics and inference times:",
        OUTPUT_DIR / "public_evaluation_summary.csv"
    )

    print(
        "\nNote: RuleTaker has test/OOD-test here; "
        "FOLIO has validation; CounterBench Backdoor has "
        "one labelled public evaluation file."
    )

    print(
        "These are deterministic reasoning engines, "
        "not conventionally trained neural models."
    )

    print(
        "\nIMPORTANT: This is a PUBLIC evaluation fallback, "
        "not a private-test submission."
    )

    return (
        public_submission,
        detailed,
        metrics_df
    )


# ============================================================
# 7. PUBLIC FALLBACK OR PRIVATE SUBMISSION
# ============================================================

if PRIVATE_TEST_PATH:

    print(
        "\nPrivate test file configured."
    )

    private_test_df = load_private_test(
        PRIVATE_TEST_PATH
    )

    submission_df = create_submission(
        private_test_df,
        SUBMISSION_PATH,
        source_name="private"
    )

    print(
        "\nsubmission.csv now contains PRIVATE-test predictions."
    )

else:

    (
        public_submission_df,
        public_detailed_df,
        public_metrics_df
    ) = build_public_evaluation_outputs()

    print(
        "\nPrivate test file not supplied yet."
    )

    print(
        "submission.csv currently contains PUBLIC evaluation predictions."
    )

    print(
        "When the instructor releases private data, "
        "set PRIVATE_TEST_PATH and rerun this cell."
    )

    print(
        "The public evaluation outputs will remain unchanged."
    )


# ============================================================
# 8. FINAL OUTPUT AUDIT
# ============================================================

expected_outputs = [

    "ruletaker_evaluation.csv",

    "counterbench_backdoor_v21_metrics.json",

    "counterbench_backdoor_v21_predictions.csv",

    "folio_z3_metrics.csv",

    "folio_parser_clean_metrics.csv",

    "public_test_predictions_detailed.csv",

    "public_evaluation_summary.csv"

]

print("\n" + "=" * 88)

print(
    "PUBLIC EXPERIMENT OUTPUT CHECK"
)

print("=" * 88)

for name in expected_outputs:

    path = OUTPUT_DIR / name

    print(
        f"{name:48s} | "
        f"{'FOUND' if path.exists() else 'NOT YET CREATED'}"
    )

print(
    "submission.csv".ljust(48),
    "|",
    "FOUND" if SUBMISSION_PATH.exists() else "NOT YET CREATED"
)

print("\n" + "=" * 88)

print(
    "SECTION 19 COMPLETED"
)

print(
    "Frozen benchmark predictions preserved."
)

print(
    "No benchmark retraining or reevaluation performed."
)

print(
    "Private submission remains pending  test data."
)

print("=" * 88)



PUBLIC DATA EVALUATION SUMMARY


,task,split,method,examples,accuracy,runtime_seconds,parser_coverage,parsed_examples,parser_failures,parser_clean_accuracy
0,ruletaker,test,forward,250,0.976000,0.132465,NaN,NaN,NaN,NaN
1,ruletaker,ood_test,forward,375,0.994667,0.173804,NaN,NaN,NaN,NaN
2,counterbench_backdoor_v21,public_labeled_evaluation,root-world abduction + intervention + prediction,200,0.900000,0.028274,NaN,NaN,NaN,NaN
3,folio_z3,validation,Z3 first-order entailment,203,0.847291,2.406531,0.945813,192.0,11.0,0.859375


Public prediction rows: 1028
Public predictions saved to: /kaggle/working/submission/submission.csv
Detailed labelled predictions: /kaggle/working/AI_Assignment_2026/outputs/public_test_predictions_detailed.csv
Metrics and inference times: /kaggle/working/AI_Assignment_2026/outputs/public_evaluation_summary.csv

Note: RuleTaker has test/OOD-test here; FOLIO has validation; CounterBench Backdoor has one labelled public evaluation file.
These are deterministic reasoning engines, not conventionally trained neural models.

IMPORTANT: This is a PUBLIC evaluation fallback, not a private-test submission.

Private test file not supplied yet.
submission.csv currently contains PUBLIC evaluation predictions.
When the instructor releases private data, set PRIVATE_TEST_PATH and rerun this cell.
The public evaluation outputs will remain unchanged.

PUBLIC EXPERIMENT OUTPUT CHECK
ruletaker_evaluation.csv                         | FOUND
counterbench_backdoor_v21_metrics.json           | FOUND
counterb

In [16]:

# ============================================================
# FINAL PUBLIC EVALUATION REPORT
# FINAL FROZEN VERSION
# ============================================================
#
# - Uses existing predictions only.

# - Reports FOLIO parser coverage and clean accuracy separately.
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []
prediction_frames = []


# ------------------------------------------------------------
# 2. RuleTaker results
# ------------------------------------------------------------

for split in ["train", "test", "ood_test"]:

    key = (split, "forward")

    if key not in RULETAKER_PREDICTIONS:
        continue

    result = RULETAKER_PREDICTIONS[key]

    actual = [bool(x) for x in result["actual"]]
    predicted = [bool(x) for x in result["predicted"]]

    if len(actual) != len(predicted):
        raise ValueError(
            f"RuleTaker {split}: prediction length mismatch."
        )

    total = len(actual)
    correct = sum(a == p for a, p in zip(actual, predicted))
    accuracy = correct / total if total else 0.0

    metric_match = results_df[
        (results_df["dataset"] == split)
        & (results_df["method"] == "forward")
    ]

    runtime = (
        float(metric_match.iloc[0]["total_elapsed_seconds"])
        if not metric_match.empty
        else float("nan")
    )

    summary_rows.append({
        "Dataset split": f"ruletaker_{split}",
        "Queries": total,
        "Correct": correct,
        "Accuracy": accuracy,
        "Time (s)": runtime
    })

    source_df = RULETAKER_DF[split]

    ids = (
        source_df["id"].astype(str).tolist()
        if "id" in source_df.columns
        else source_df.index.astype(str).tolist()
    )

    if len(ids) != total:
        raise ValueError(
            f"RuleTaker {split}: ID/prediction length mismatch."
        )

    prediction_frames.append(pd.DataFrame({
        "id": [f"ruletaker_{split}_{x}" for x in ids],
        "predicted_label": [
            "True" if p else "False" for p in predicted
        ]
    }))


# ------------------------------------------------------------
# 3. CounterBench Backdoor V2.1
# ------------------------------------------------------------

if (
    "backdoor_v21_predictions" in globals()
    and not backdoor_v21_predictions.empty
):

    cb = backdoor_v21_predictions.copy()

    total = len(cb)
    correct = int(cb["correct"].astype(bool).sum())
    accuracy = correct / total if total else 0.0

    runtime = float(
        metrics_v21.get("runtime_seconds", float("nan"))
    )

    summary_rows.append({
        "Dataset split": "counterbench_backdoor_v21",
        "Queries": total,
        "Correct": correct,
        "Accuracy": accuracy,
        "Time (s)": runtime
    })

    prediction_frames.append(pd.DataFrame({
        "id": [
            f"counterbench_backdoor_{x}"
            for x in cb["question_id"].astype(str)
        ],
        "predicted_label": [
            "True" if p else "False"
            for p in cb["predicted"].astype(bool)
        ]
    }))


# ------------------------------------------------------------
# 4. FOLIO train and validation
# ------------------------------------------------------------

folio_parser_observations = []

for split, metric_var, prediction_var in [
    ("train", "train_metrics", "train_predictions"),
    ("validation", "validation_metrics", "validation_predictions")
]:

    if (
        metric_var not in globals()
        or prediction_var not in globals()
    ):
        continue

    metrics = globals()[metric_var]
    predictions = globals()[prediction_var].copy()

    if predictions.empty:
        continue

    total = len(predictions)

    correct = int(
        predictions["correct"].astype(bool).sum()
    )

    accuracy = correct / total if total else 0.0

    runtime = float(
        metrics.get("runtime_seconds", float("nan"))
    )

    # Parser-clean evaluation.
    if "solver_status" in predictions.columns:

        parser_error_mask = (
            predictions["solver_status"]
            == "parse_or_translation_error"
        )

        parser_failures = int(parser_error_mask.sum())

    else:

        parser_error_mask = pd.Series(
            False,
            index=predictions.index
        )

        parser_failures = 0

    parsed_count = total - parser_failures

    parser_coverage = (
        parsed_count / total if total else 0.0
    )

    clean_predictions = predictions.loc[
        ~parser_error_mask
    ]

    clean_accuracy = (
        clean_predictions["correct"].astype(bool).mean()
        if not clean_predictions.empty
        else float("nan")
    )

    summary_rows.append({
        "Dataset split": f"FOLIO_{split}",
        "Queries": total,
        "Correct": correct,
        "Accuracy": accuracy,
        "Time (s)": runtime
    })

    folio_parser_observations.append({
        "split": split,
        "total": total,
        "parsed": parsed_count,
        "parser_failures": parser_failures,
        "coverage": parser_coverage,
        "clean_accuracy": clean_accuracy
    })

    # IMPORTANT:
    # Only common columns are included in the combined
    # prediction file. No solver_status column here.
    prediction_frames.append(pd.DataFrame({
        "id": [
            f"folio_{split}_{x}"
            for x in predictions["example_id"].astype(str)
        ],
        "predicted_label": [
            "True" if p else "False"
            for p in predictions["predicted_label"].astype(bool)
        ]
    }))


# ------------------------------------------------------------
# 5. Build summary table
# ------------------------------------------------------------

if not summary_rows:
    raise RuntimeError(
        "No evaluation results found. "
        "Run the evaluation cells first."
    )

summary_df = pd.DataFrame(summary_rows)

total_queries = int(summary_df["Queries"].sum())
total_correct = int(summary_df["Correct"].sum())

overall_accuracy = (
    total_correct / total_queries
    if total_queries else 0.0
)

total_runtime = summary_df["Time (s)"].sum(
    min_count=1
)


# ------------------------------------------------------------
# 6. Display  report
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("FINAL PUBLIC DATA EVALUATION SUMMARY")
print("=" * 88)

header = (
    f"{'Dataset split':<34}"
    f"{'Queries':>10}"
    f"{'Correct':>10}"
    f"{'Accuracy':>12}"
    f"{'Time (s)':>12}"
)

print("\n" + header)
print("-" * 88)

for _, row in summary_df.iterrows():

    print(
        f"{row['Dataset split']:<34}"
        f"{int(row['Queries']):>10}"
        f"{int(row['Correct']):>10}"
        f"{row['Accuracy']:>11.2%}"
        f"{row['Time (s)']:>12.4f}"
    )

print("-" * 88)

print(
    f"{'OVERALL TOTAL':<34}"
    f"{total_queries:>10}"
    f"{total_correct:>10}"
    f"{overall_accuracy:>11.2%}"
    f"{total_runtime:>12.4f}"
)

print("\n" + "=" * 88)


# ------------------------------------------------------------
# 7. Display notebook table
# ------------------------------------------------------------

report_display = summary_df.copy()

report_display["Queries"] = (
    report_display["Queries"].map("{:,}".format)
)

report_display["Correct"] = (
    report_display["Correct"].map("{:,}".format)
)

report_display["Accuracy"] = (
    report_display["Accuracy"].map(
        lambda x: f"{x:.2%}"
    )
)

report_display["Time (s)"] = (
    report_display["Time (s)"].map(
        lambda x: f"{x:.4f}"
        if pd.notna(x) else "N/A"
    )
)

total_display = pd.DataFrame([{
    "Dataset split": "Total",
    "Queries": f"{total_queries:,}",
    "Correct": f"{total_correct:,}",
    "Accuracy": f"{overall_accuracy:.2%}",
    "Time (s)": (
        f"{total_runtime:.4f}"
        if pd.notna(total_runtime) else "N/A"
    )
}])

report_display = pd.concat(
    [report_display, total_display],
    ignore_index=True
)

print("\nResults on the Public Datasets")

display(report_display)


# ------------------------------------------------------------
# 8. Observations
# ------------------------------------------------------------

print("\nOBSERVATIONS\n")

print(
    "1. RuleTaker: Results are reported separately for "
    "training, test and OOD-test splits."
)

if (
    "backdoor_v21_predictions" in globals()
    and not backdoor_v21_predictions.empty
):

    print(
        "2. CounterBench Backdoor V2.1: "
        f"{int(backdoor_v21_predictions['correct'].sum())}/"
        f"{len(backdoor_v21_predictions)} correct "
        f"({backdoor_v21_predictions['correct'].mean():.2%})."
    )

if folio_parser_observations:

    print(
        "3. FOLIO: Parser coverage and parser-clean accuracy "
        "are reported separately from overall accuracy."
    )

    for item in folio_parser_observations:

        print(
            f"   {item['split'].capitalize()}: "
            f"{item['parsed']}/{item['total']} parsed "
            f"({item['coverage']:.2%} coverage), "
            f"clean accuracy "
            f"{item['clean_accuracy']:.2%}, "
            f"{item['parser_failures']} parser failures."
        )

print(
    "\n4. The combined overall accuracy is descriptive only. "
    "It mixes different datasets and evaluation splits; "
    "individual benchmark results should be used for "
    "scientific comparisons."
)


# ------------------------------------------------------------
# 9. Save summary CSV
# ------------------------------------------------------------

summary_path = (
    OUTPUT_DIR / "final_public_evaluation_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\nSaved:", summary_path)


# ------------------------------------------------------------
# 10. Save combined public predictions
# ------------------------------------------------------------

if prediction_frames:

    public_predictions = pd.concat(
        prediction_frames,
        ignore_index=True
    )

    # Keep exactly two common columns.
    public_predictions = public_predictions[
        ["id", "predicted_label"]
    ]

    if public_predictions["id"].duplicated().any():

        raise ValueError(
            "Duplicate IDs detected in public predictions."
        )

    if public_predictions.isna().any().any():

        raise ValueError(
            "Missing values detected in public predictions."
        )

    if not set(
        public_predictions["predicted_label"].unique()
    ).issubset({"True", "False"}):

        raise ValueError(
            "Invalid labels detected in public predictions."
        )

    predictions_path = (
        OUTPUT_DIR / "public_predictions_all_splits.csv"
    )

    public_predictions.to_csv(
        predictions_path,
        index=False
    )

    print(
        f"\nSaved {len(public_predictions)} public predictions."
    )

    print(
        "Saved:",
        predictions_path
    )

else:

    print("\nNo prediction records available.")


# ------------------------------------------------------------
# 11. Save machine-readable JSON report
# ------------------------------------------------------------

overall_report = {

    "total_public_examples": total_queries,

    "total_correct": total_correct,

    "descriptive_combined_accuracy": overall_accuracy,

    "total_execution_time_seconds": (
        float(total_runtime)
        if pd.notna(total_runtime)
        else None
    ),

    "warning": (
        "Combined accuracy is descriptive only and mixes "
        "different datasets and splits."
    ),

    "dataset_results": summary_rows,

    "folio_parser_observations": folio_parser_observations

}

json_path = (
    OUTPUT_DIR / "final_public_evaluation_metrics.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        overall_report,
        file,
        indent=4
    )

print(
    "Saved:",
    json_path
)


# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("FINAL PUBLIC EVALUATION REPORT GENERATED")
print("=" * 88)

print(
    f"Total public examples : {total_queries:,}"
)

print(
    f"Total correct         : {total_correct:,}"
)

print(
    f"Descriptive accuracy  : {overall_accuracy:.2%}"
)

print(
    "\nsubmission.csv was NOT modified by this cell."
)

print(
    "Section 19 remains responsible for public fallback "
    "and future private submission."
)

print(
    "\nAll existing benchmark predictions were reused."
)

print("=" * 88)



FINAL PUBLIC DATA EVALUATION SUMMARY

Dataset split                        Queries   Correct    Accuracy    Time (s)
----------------------------------------------------------------------------------------
ruletaker_train                         1000       983     98.30%      0.4740
ruletaker_test                           250       244     97.60%      0.1325
ruletaker_ood_test                       375       373     99.47%      0.1738
counterbench_backdoor_v21                200       180     90.00%      0.0283
FOLIO_train                             1001       814     81.32%     10.0333
FOLIO_validation                         203       172     84.73%      2.4065
----------------------------------------------------------------------------------------
OVERALL TOTAL                           3029      2766     91.32%     13.2484


Results on the Public Datasets


,Dataset split,Queries,Correct,Accuracy,Time (s)
0,ruletaker_train,"1,000",983,98.30%,0.4740
1,ruletaker_test,250,244,97.60%,0.1325
2,ruletaker_ood_test,375,373,99.47%,0.1738
3,counterbench_backdoor_v21,200,180,90.00%,0.0283
4,FOLIO_train,"1,001",814,81.32%,10.0333
5,FOLIO_validation,203,172,84.73%,2.4065
6,Total,"3,029","2,766",91.32%,13.2484



OBSERVATIONS

1. RuleTaker: Results are reported separately for training, test and OOD-test splits.
2. CounterBench Backdoor V2.1: 180/200 correct (90.00%).
3. FOLIO: Parser coverage and parser-clean accuracy are reported separately from overall accuracy.
   Train: 896/1001 parsed (89.51% coverage), clean accuracy 83.15%, 105 parser failures.
   Validation: 192/203 parsed (94.58% coverage), clean accuracy 85.94%, 11 parser failures.

4. The combined overall accuracy is descriptive only. It mixes different datasets and evaluation splits; individual benchmark results should be used for scientific comparisons.

Saved: /kaggle/working/AI_Assignment_2026/outputs/final_public_evaluation_summary.csv

Saved 3029 public predictions.
Saved: /kaggle/working/AI_Assignment_2026/outputs/public_predictions_all_splits.csv
Saved: /kaggle/working/AI_Assignment_2026/outputs/final_public_evaluation_metrics.json

FINAL PUBLIC EVALUATION REPORT GENERATED
Total public examples : 3,029
Total correct         

# ============================================================
# SECTION 19 — OPTIONAL PRIVATE-TEST INFERENCE + SUBMISSION.CSV
# ============================================================

from pathlib import Path
import json
import pandas as pd
import time

# Set this to the instructor-provided private test file path in Kaggle.
# Supported input formats: CSV, JSONL, JSON.
# Do NOT point this at a public train/validation file.
PRIVATE_TEST_PATH = None  # Example: "/kaggle/input/<competition-folder>/test.csv"

SUBMISSION_DIR = Path("/kaggle/working/submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

def load_private_test(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Private test file not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".jl"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    raise ValueError("Use a .csv, .jsonl, or .json private test file.")

def _as_bool_prediction(value):
    if isinstance(value, dict):
        value = value.get("prediction")
    if value is None:
        raise ValueError("Reasoning engine returned no prediction.")
    return bool(value)

def predict_private_record(row):
    """
    Dispatch a row to the classical reasoning engine based on its fields.
    Accepted schemas:
      RuleTaker: context, statement
      CounterBench: given_info, question
      FOLIO: premises-FOL, conclusion-FOL
    An optional task column can be 'ruletaker', 'counterbench', or 'folio'.
    """
    record = row.to_dict() if isinstance(row, pd.Series) else dict(row)
    task = str(record.get("task", "")).strip().lower()

    if not task:
        if {"context", "statement"}.issubset(record):
            task = "ruletaker"
        elif {"given_info", "question"}.issubset(record):
            task = "counterbench"
        elif {"premises-FOL", "conclusion-FOL"}.issubset(record):
            task = "folio"
        else:
            raise ValueError(
                "Cannot infer task. Provide task or one of these field pairs: "
                "(context, statement), (given_info, question), "
                "(premises-FOL, conclusion-FOL)."
            )

    if task in {"ruletaker", "rule_taker", "rule-taker"}:
        result = predict_ruletaker(
            record["context"], record["statement"], method="forward"
        )
        return _as_bool_prediction(result)

    if task in {"counterbench", "counterbench_backdoor", "backdoor"}:
        # This uses the separate Backdoor V2.1 causal counterfactual engine.
        # The frozen Alpha PBCI V1.0 result and original V2 baseline are not altered.
        return _as_bool_prediction(
            cb21_answer(record["given_info"], record["question"])
        )

    if task in {"folio", "folio_z3"}:
        prover = FOLIOProver(timeout_ms=8000, fuzzy_alignment=True)
        predicted, solver_status = prover.prove_folio_detailed(
            record["premises-FOL"], record["conclusion-FOL"]
        )
        return bool(predicted)

    raise ValueError(f"Unsupported task value: {task!r}")

def create_submission(test_df, output_path):
    if test_df.empty:
        raise ValueError("Private test dataset is empty.")

    id_candidates = ["id", "example_id", "question_id"]
    id_col = next((c for c in id_candidates if c in test_df.columns), None)
    if id_col is None:
        raise ValueError(
            "Private test file must contain an ID column named one of: "
            + ", ".join(id_candidates)
        )
    if test_df[id_col].isna().any():
        raise ValueError("Private test file contains missing IDs.")
    if test_df[id_col].duplicated().any():
        raise ValueError("Private test file contains duplicate IDs.")

    start = time.perf_counter()
    predictions = []
    errors = []
    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Private inference"):
        try:
            predictions.append(predict_private_record(row))
        except Exception as exc:
            errors.append({
                "row_index": int(index),
                "id": row[id_col],
                "error": f"{type(exc).__name__}: {exc}",
            })
            predictions.append(None)

    elapsed = time.perf_counter() - start
    if errors:
        error_path = Path(output_path).with_name("submission_errors.csv")
        pd.DataFrame(errors).to_csv(error_path, index=False)
        raise RuntimeError(
            f"{len(errors)} rows could not be predicted. No submission was written. "
            f"Inspect {error_path}; fix parser/schema issues and rerun."
        )

    submission = pd.DataFrame({
        "id": test_df[id_col].values,
        "predicted_label": ["True" if p else "False" for p in predictions],
    })
    if len(submission) != len(test_df):
        raise AssertionError("Submission row count does not match private test input.")
    if submission["id"].duplicated().any() or submission.isna().any().any():
        raise AssertionError("Submission contains duplicate IDs or missing values.")
    if not set(submission["predicted_label"].unique()).issubset({"True", "False"}):
        raise AssertionError("Submission contains an invalid label.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)

    print("\nPRIVATE-TEST INFERENCE SUMMARY")
    print("Rows:", len(submission))
    print(f"Total inference time: {elapsed:.6f} seconds")
    print(f"Average per example: {elapsed / len(submission):.8f} seconds")
    print("Saved:", output_path)
    print("Columns:", submission.columns.tolist())
    display(submission.head())
    return submission

if PRIVATE_TEST_PATH:
    private_test_df = load_private_test(PRIVATE_TEST_PATH)
    submission_df = create_submission(
        private_test_df,
        SUBMISSION_DIR / "submission.csv"
    )
else:
    print(
        "Private test file not configured. No submission.csv has been fabricated.\n"
        "After the instructor supplies the private test file, set PRIVATE_TEST_PATH "
        "above and rerun this cell. Expected output: /kaggle/working/submission/submission.csv"
    )

# Final public-experiment output audit (does not overwrite any benchmark file).
expected_outputs = [
    "counterbench_backdoor_v21_metrics.json",
    "counterbench_backdoor_v21_predictions.csv",
    "folio_z3_metrics.csv",
]
print("\nPUBLIC EXPERIMENT OUTPUT CHECK")
for name in expected_outputs:
    path = OUTPUT_DIR / name
    print(f"{name:48s} | {'FOUND' if path.exists() else 'NOT YET CREATED'}")

In [17]:
print("TEst")

TEst
